# 🎨 Pyxel Canvas
### 100 Stunning 2D & 3D Patterns in a Single Jupyter Notebook

---

Welcome to **Pyxel Canvas** — a fully interactive visual engine and gallery system.
Select a category, pick a pattern, adjust the controls, and click **RENDER**.

> **Tip:** Run all cells from top to bottom on first launch. Section 0 will verify all dependencies.

---
## Section 0 — Environment Setup
Verify all required libraries are available.

In [ ]:
# ── Section 0: Environment Setup ─────────────────────────────────
import importlib
from pathlib import Path

# Root path — all other paths are relative to this
NOTEBOOK_ROOT = Path.cwd()
print(f"📁 NOTEBOOK_ROOT = {NOTEBOOK_ROOT}")

# Required packages: (pip_name, import_name)
_REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("Pillow", "PIL"),
    ("ipywidgets", "ipywidgets"),
    ("scipy", "scipy"),
    ("tqdm", "tqdm"),
]

print("\n🔧 Checking dependencies...\n")
_missing = []
for pip_name, import_name in _REQUIRED:
    try:
        importlib.import_module(import_name)
        print(f"  ✅ {pip_name} — available")
    except ImportError:
        print(f"  ❌ {pip_name} — MISSING")
        _missing.append(pip_name)

if _missing:
    print(f"\n⚠️  Missing packages: {', '.join(_missing)}")
    print("   Install via MSYS2:  pacman -S mingw-w64-ucrt-x86_64-python-<name>")
    print("   Or via pip:         python -m pip install <name>")
else:
    # Create output directories
    for d in ["exports", "assets", "shaders"]:
        (NOTEBOOK_ROOT / d).mkdir(exist_ok=True)
    print("\n✨ Environment ready!")

---
## Section 1 — Core Engine
Load pattern registry, engines, and utilities into the notebook namespace.

In [ ]:
# ── Section 1: Core Engine ───────────────────────────────────────
import sys
from pathlib import Path

# Make sure our package is importable
_engine_root = str(Path.cwd().parent) if Path.cwd().name == 'visual_engine' else str(Path.cwd())
if _engine_root not in sys.path:
    sys.path.insert(0, _engine_root)

_ve_root = str(Path.cwd()) if Path.cwd().name == 'visual_engine' else str(Path.cwd() / 'visual_engine')
if _ve_root not in sys.path:
    sys.path.insert(0, _ve_root)

# Import the pattern registry
from patterns import PATTERNS, CATEGORIES
from engines import PALETTES
from engines.color_utils import ColorUtils
from utils.export import export_png

print(f"🎨 Loaded {len(PATTERNS)} patterns across {len(CATEGORIES)} categories")
for cat, pats in CATEGORIES.items():
    print(f"   • {cat}: {len(pats)} patterns")

---
## Section 2 — UI Controls
Interactive gallery interface — select a category and pattern, adjust controls, render.

In [ ]:
# ── Section 2: UI Controls ───────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# ── Caches ────────────────────────────────────────────────────────
_render_cache   = {}   # renderer instances, keyed by pattern name
_controls_cache = {}   # widget lists, keyed by pattern name — same instances reused everywhere

def _get_renderer(name):
    if name not in _render_cache:
        _render_cache[name] = PATTERNS[name]
    return _render_cache[name]

def _get_controls(name):
    """Return the cached control widgets for a pattern (created once, reused forever)."""
    if name not in _controls_cache:
        _controls_cache[name] = _get_renderer(name).get_controls()
    return _controls_cache[name]

# ── Widgets ────────────────────────────────────────────────────────

category_dd = widgets.Dropdown(
    options=list(CATEGORIES.keys()),
    value=list(CATEGORIES.keys())[0],
    description='Category:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='320px'),
)

pattern_dd = widgets.Dropdown(
    options=CATEGORIES[category_dd.value],
    description='Pattern:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='320px'),
)

palette_dd = widgets.Dropdown(
    options=list(PALETTES.keys()),
    value='Inferno',
    description='Palette:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
)

speed_slider = widgets.FloatSlider(
    value=1.0, min=0.1, max=5.0, step=0.1,
    description='Speed:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
    readout_format='.1f',
)

resolution_dd = widgets.Dropdown(
    options=['Low', 'Medium', 'High'],
    value='Low',
    description='Resolution:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
)

render_btn = widgets.Button(
    description='🎨 RENDER',
    button_style='success',
    layout=widgets.Layout(width='160px', height='40px'),
    style={'font_weight': 'bold'},
)

export_btn = widgets.Button(
    description='💾 EXPORT',
    button_style='info',
    layout=widgets.Layout(width='160px', height='40px'),
    style={'font_weight': 'bold'},
)

# VBox instead of Output — children replacement is atomic, no clear_output race conditions
pattern_controls_area = widgets.VBox(
    layout=widgets.Layout(
        border='1px solid #333', min_height='40px', padding='8px', margin='4px 0',
    )
)

render_output = widgets.Output(layout=widgets.Layout(
    border='1px solid #444', min_height='200px', padding='8px',
))

status_label = widgets.HTML(
    value='<i style="color:#888;">Select a pattern and click RENDER</i>',
    layout=widgets.Layout(margin='4px 0'),
)

# ── Callbacks ──────────────────────────────────────────────────────

def _on_category_change(change):
    """Switch category — update pattern_dd without double-firing _on_pattern_change."""
    pattern_dd.unobserve(_on_pattern_change, names='value')
    pattern_dd.options = CATEGORIES[change['new']]
    pattern_dd.value   = CATEGORIES[change['new']][0]
    pattern_dd.observe(_on_pattern_change, names='value')
    _on_pattern_change({'new': pattern_dd.value})   # fire exactly once

def _on_pattern_change(change):
    """Swap the controls panel — atomic VBox.children replace, no flicker or doubling."""
    controls = _get_controls(change['new'])
    pattern_controls_area.children = (
        controls if controls
        else [widgets.Label('No pattern-specific controls.')]
    )

_last_fig = [None]

def _on_render(btn):
    """Render selected pattern — plt.show() inside render() is the only display event."""
    with render_output:
        clear_output(wait=True)
        name = pattern_dd.value
        status_label.value = f'<b style="color:#4fc3f7;">⏳ Rendering: {name}...</b>'
        renderer = _get_renderer(name)

        # Read values from the DISPLAYED controls (same cached instances the user adjusted).
        extra_kwargs = {}
        for ctrl in _get_controls(name):
            if hasattr(ctrl, 'description') and hasattr(ctrl, 'value'):
                key = ctrl.description.strip(':').strip().lower().replace(' ', '_')
                extra_kwargs[key] = ctrl.value

        try:
            # render() calls plt.show() then plt.close(fig) — that is the one and only
            # display event.  No sink, no manual display() call needed here.
            renderer.render(
                resolution=resolution_dd.value,
                palette=palette_dd.value,
                speed=speed_slider.value,
                **extra_kwargs,
            )
            # renderer._fig is assigned by _create_figure() and the Python object
            # remains valid for savefig() even after plt.close() deregisters it.
            _last_fig[0] = renderer._fig
            status_label.value = f'<b style="color:#81c784;">✅ Rendered: {name}</b>'
        except Exception as e:
            status_label.value = f'<b style="color:#e57373;">❌ Error: {e}</b>'
            import traceback
            traceback.print_exc()

def _on_export(btn):
    """Export the last rendered figure to PNG."""
    if _last_fig[0] is not None:
        path = export_png(_last_fig[0], pattern_dd.value)
        status_label.value = f'<b style="color:#81c784;">💾 Exported → {path}</b>'
    else:
        status_label.value = '<b style="color:#ffb74d;">⚠️ Nothing to export — render a pattern first.</b>'

# Wire up callbacks
category_dd.observe(_on_category_change, names='value')
pattern_dd.observe(_on_pattern_change,   names='value')
render_btn.on_click(_on_render)
export_btn.on_click(_on_export)

# ── Layout ─────────────────────────────────────────────────────────

header = widgets.HTML(value='''
    <div style="background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);
                padding:16px 24px; border-radius:12px; margin-bottom:12px;">
        <h2 style="color:#e0e0e0; margin:0;">🎨 Pyxel Canvas</h2>
        <p style="color:#888; margin:4px 0 0;">Select a category → pick a pattern → click RENDER</p>
    </div>
''')

selectors_row = widgets.HBox([category_dd, pattern_dd],
                              layout=widgets.Layout(gap='12px'))
controls_row  = widgets.HBox([palette_dd, speed_slider, resolution_dd],
                              layout=widgets.Layout(gap='12px'))
buttons_row   = widgets.HBox([render_btn, export_btn],
                              layout=widgets.Layout(gap='12px'))

ui = widgets.VBox([
    header,
    selectors_row,
    controls_row,
    pattern_controls_area,
    buttons_row,
    status_label,
    render_output,
], layout=widgets.Layout(padding='8px'))

display(ui)
_on_pattern_change({'new': pattern_dd.value})

---
## Section 3 — Render Section
The RENDER button above triggers rendering. No separate cell needed — the UI handles it.

---

## Section 4 — Pattern Implementations

All 100 patterns are implemented as classes in the `patterns/` module files.
Concept Notes for each completed pattern are listed below.

### How This Works — Mandelbrot Fractal Explorer

**Core Idea**  
The Mandelbrot set is the set of complex numbers $c$ for which the sequence produced by iterating $z_{n+1} = z_n^2 + c$ (starting from $z_0 = 0$) remains bounded — that is, never escapes to infinity. Points inside the set are colored black; points outside are colored by how quickly they escape, producing the iconic fractal boundary.

**Mathematics**  
The iteration is:
$$z_{n+1} = z_n^2 + c, \quad z_0 = 0, \quad c \in \mathbb{C}$$
Each pixel maps to a value of $c = x + iy$, where $x$ and $y$ are the real and imaginary axes. The escape condition is $|z_n| > 2$. The `max_iter` control sets the iteration ceiling: higher values reveal finer boundary detail at the cost of computation time. Smooth coloring uses the formula $n_{\text{smooth}} = n + 1 - \log_2(\log_2|z_n|)$ to eliminate banding.

**Logic in the Code**  
1. A 2D numpy array of complex numbers is constructed via `meshgrid` over the viewport defined by `center` and `zoom`.  
2. The iteration $z = z^2 + c$ is applied element-wise using numpy broadcasting — no Python loop over pixels.  
3. An escape mask tracks which pixels have crossed $|z| > 2$. Those pixels are frozen and excluded from subsequent steps.  
4. The smooth escape-count array is passed through a `ColorUtils` colormap to produce the final image.

**Interesting Properties**  
The Mandelbrot set is infinitely self-similar: zooming into any part of the boundary reveals structures that resemble — but are not identical to — the whole set. The cardioid shape of the main body is the locus of $c$ values where $z$ converges to a fixed point, while the circular bulb to its left corresponds to period-2 orbits.

### How This Works — Julia Set Animator

**Core Idea**  
A Julia set is the companion fractal to the Mandelbrot set. Instead of varying $c$ per pixel and starting from $z_0 = 0$, a Julia set fixes $c$ and varies the starting point $z_0$ across the complex plane. Each value of $c$ produces a different Julia set — connected (filled) when $c$ is inside the Mandelbrot set, and a Cantor dust when $c$ is outside.

**Mathematics**  
$$z_{n+1} = z_n^2 + c, \quad z_0 = x + iy \text{ (varies per pixel)}, \quad c \text{ is fixed}$$
The escape condition and smooth coloring are identical to Mandelbrot: $|z_n| > 2$ and $n_{\text{smooth}} = n + 1 - \log_2(\log_2|z_n|)$. The `c_real` and `c_imag` sliders control the fixed constant, allowing exploration of radically different fractal shapes.

**Logic in the Code**  
1. A complex grid of $z_0$ values is built from the viewport extent.  
2. The same vectorized escape-time loop runs, but with a constant $c$ added at each step.  
3. The resulting escape array is rendered through the selected colormap.

**Interesting Properties**  
Moving the $c$ parameter along the boundary of the Mandelbrot set produces Julia sets that transition from connected to disconnected — the so-called Douady rabbit ($c \approx -0.123 + 0.745i$) and the dendrite ($c = i$) are famous examples. The Mandelbrot set is, in fact, a map of all possible Julia set topologies.

### How This Works — Sierpinski Triangle

**Core Idea**  
The Sierpinski triangle is a self-similar fractal formed by recursively removing the central inverted triangle from an equilateral triangle. After infinite iterations the remaining area is zero, but the structure retains infinite detail. It appears naturally in Pascal's triangle (odd entries) and cellular automata (Rule 90).

**Mathematics**  
At each recursion level, every triangle is replaced by three sub-triangles at half scale, positioned at the three corners. The fractal dimension is:
$$D = \frac{\log 3}{\log 2} \approx 1.585$$
At depth $d$, the number of filled triangles is $3^d$ and each has side length $2^{-d}$ of the original. The `depth` control directly sets $d$.

**Logic in the Code**  
1. Start with the three vertices of an equilateral triangle.  
2. Recursively subdivide: compute midpoints of each edge, forming three corner triangles (discarding the center).  
3. At the base case ($d = 0$), store the triangle as a `[v0, v1, v2]` list.  
4. All triangles are drawn as filled `Polygon` patches with colors sampled from a gradient.

**Interesting Properties**  
The Sierpinski triangle can also be generated by the chaos game: repeatedly pick a random vertex and move halfway toward it from the current point. Despite the randomness, the attractor is deterministically the Sierpinski triangle.

### How This Works — Koch Snowflake

**Core Idea**  
The Koch snowflake is constructed by repeatedly replacing the middle third of every line segment with an equilateral triangular bump. Starting from an equilateral triangle, this process produces a closed curve of infinite length enclosing a finite area — a key early example of a fractal curve.

**Mathematics**  
Each iteration multiplies the number of segments by 4 and divides each segment's length by 3. After $n$ iterations:
$$L_n = L_0 \cdot \left(\frac{4}{3}\right)^n \to \infty, \qquad A_n = A_0 \cdot \frac{8}{5}\left(1 - \frac{3}{9^n}\right)$$
The fractal dimension of the Koch curve is $D = \log 4 / \log 3 \approx 1.262$. The `iterations` slider controls $n$.

**Logic in the Code**  
1. Define three vertices of an equilateral triangle as the snowflake base.  
2. For each edge, recursively subdivide: split into thirds, compute the peak point by rotating the middle third by $60°$ using a 2D rotation.  
3. Collect all points and draw each line segment with a color from the gradient, producing a rainbow Koch snowflake.

**Interesting Properties**  
The Koch snowflake is nowhere differentiable — it has no tangent at any point. It was one of the first curves shown to be continuous everywhere but differentiable nowhere, predating Mandelbrot's formalization of fractal geometry by decades.

### How This Works — Penrose Tiling

**Core Idea**  
A Penrose tiling is an aperiodic tiling — it fills the plane without gaps or overlaps, yet never repeats. Discovered by Roger Penrose in 1974, the P3 variant uses two rhombus shapes (thin and thick) derived from Robinson triangles. The pattern exhibits five-fold rotational symmetry, which is forbidden in periodic crystals.

**Mathematics**  
The construction uses the golden ratio $\varphi = (1 + \sqrt{5})/2 \approx 1.618$. Robinson triangle deflation splits each triangle into smaller ones using subdivision points at $1/\varphi$ along edges:
$$P = A + \frac{B - A}{\varphi}, \qquad Q = B + \frac{A - B}{\varphi}, \qquad R = B + \frac{C - B}{\varphi}$$
Thin (acute) triangles split into 2 sub-triangles; thick (obtuse) triangles split into 3. The `generations` control sets the number of deflation rounds.

**Logic in the Code**  
1. Start with a sun configuration: 10 isosceles triangles arranged in a decagon (alternating orientations).  
2. For each generation, apply the deflation rules — thin → 2 children, thick → 3 children.  
3. Draw all resulting triangles as filled polygons, using two palette colors to distinguish thin from thick rhombi.

**Interesting Properties**  
Penrose tilings are quasicrystals in 2D: they have long-range order (sharp diffraction peaks with five-fold symmetry) but no translational periodicity. The ratio of thick to thin rhombi converges to $\varphi$ as the tiling grows — the golden ratio appears at every scale.

### How This Works — Voronoi Diagram

**Core Idea**
A Voronoi diagram partitions a plane into regions based on proximity to a set of seed points. Each region — called a Voronoi cell — contains every point closer to its seed than to any other. The result is a natural tessellation found everywhere in nature: giraffe skin, soap bubble clusters, and cellular tissue.

**Mathematics**
For seeds $P = \{p_1, \ldots, p_n\}$, the Voronoi cell of $p_i$ is:
$$V(p_i) = \{ x \in \mathbb{R}^2 \mid \|x - p_i\| \leq \|x - p_j\| \;\forall\, j \neq i \}$$
The boundary between two adjacent cells is the perpendicular bisector of the segment joining their seeds. The *Points* slider controls $n$; more seeds produce smaller, more uniform cells.

**Logic in the Code**
1. $n$ random seeds are placed in $[0,1]^2$ using a seeded NumPy RNG for reproducibility.
2. A `scipy.spatial.cKDTree` spatial index is built on the seeds.
3. The canvas is rasterized as a flat grid of $\text{res}^2$ pixels; one `tree.query` call assigns the nearest seed to every pixel simultaneously — no Python loop over pixels.
4. The integer assignment array indexes into the palette gradient to produce the final RGB image rendered via `imshow`.

**Interesting Properties**
Voronoi diagrams are the geometric dual of Delaunay triangulations: connecting adjacent seeds whose cells share a border produces the Delaunay graph, which maximises the minimum triangle angle across the plane. Lloyd's algorithm — iteratively replacing each seed with its cell centroid — converges toward perfectly uniform hexagonal packing.

### How This Works — Fibonacci Spiral

**Core Idea**
The Fibonacci spiral places $n$ points by rotating each successive point by the golden angle — the strategy used by sunflower seeds, pine cones, and pineapple scales to pack the maximum number of seeds per unit area without gaps or clumping.

**Mathematics**
Each seed $i$ is placed at polar coordinates:
$$r_i = \sqrt{i/n}, \qquad \theta_i = i \cdot \phi_g$$
where the **golden angle** is:
$$\phi_g = \pi(3 - \sqrt{5}) \approx 137.508°$$
The irrational nature of $\phi_g$ — derived from the golden ratio $\varphi = \tfrac{1+\sqrt{5}}{2}$ — ensures no two seeds ever land on the same radial spoke, so every new seed fills the largest visible gap.

**Logic in the Code**
1. Indices $i = 0, \ldots, n{-}1$ are generated as a NumPy array.
2. Radii are $\sqrt{i/n}$ (uniform area distribution) and angles are $i \cdot \phi_g$.
3. Cartesian coordinates follow from standard polar conversion; the palette gradient maps $i$ linearly to colour so inner and outer seeds receive different hues.
4. `ax.scatter` renders all $n$ points in a single vectorised call.

**Interesting Properties**
Plotting consecutive Fibonacci-number multiples of seeds reveals the clockwise and counter-clockwise spiral arms visible in real sunflowers. Displacing the angle by even 0.1° away from $\phi_g$ causes the pattern to collapse into coarse radial spokes — demonstrating why the golden angle is the uniquely optimal packing angle.

### How This Works — Dragon Curve

**Core Idea**
The Dragon Curve is a fractal obtained by repeatedly folding a strip of paper in half in the same direction, then unfolding it so every crease stands at 90°. After $n$ folds the unfolded boundary traces a self-avoiding curve that tiles the plane when four copies are assembled around a common point.

**Mathematics**
The turn sequence $T_n$ — recording right (R) or left (L) turns at each step — obeys the recurrence:
$$T_{n+1} = T_n \;\|\; [\text{R}] \;\|\; \overline{T_n^{\;\text{rev}}}$$
where $\overline{\cdot}$ flips all turns (R$\leftrightarrow$L) and $\cdot^{\text{rev}}$ reverses the list. Starting from $T_1 = [\text{R}]$, iteration $n$ yields $2^n - 1$ turns and $2^n$ unit segments. The Hausdorff dimension of the Dragon Curve boundary is $\tfrac{\log 4}{\log 3} \approx 1.52$.

**Logic in the Code**
1. The turn list is built iteratively: each pass appends the current list, a right turn, then the reversed-and-negated list.
2. A direction index (0–3, cycling right/up/left/down) is updated for each turn and used to step the path one unit forward.
3. The path array is converted into a `(N{-}1, 2, 2)` segment array and passed to `LineCollection` — all $2^n{-}1$ segments drawn in a single renderer call, essential for $n \geq 12$.

**Interesting Properties**
Despite its jagged appearance the Dragon Curve never self-intersects, yet in the limit $n \to \infty$ it fills a bounded fractal region completely. Four copies rotated by multiples of 90° tile the plane perfectly — a property exploited by Jurassic Park's book cover art and numerous fractal installations.

### How This Works — Hilbert Curve

**Core Idea**
The Hilbert Curve is a continuous space-filling curve that visits every cell of an $n \times n$ grid exactly once while moving only to directly adjacent cells. Invented by David Hilbert in 1891, it preserves spatial locality so well that nearby points in 2D remain nearby along the 1D index — making it invaluable for database locality, image compression, and CPU cache optimisation.

**Mathematics**
At order $k$ the curve covers a $2^k \times 2^k$ grid visiting all $4^k$ cells. Each order is built from four rotated/reflected copies of the previous order. The index-to-coordinate mapping decodes $d \in [0,4^k)$ by reading successive 2-bit pairs $(r_x, r_y)$:
$$r_x = \bigl\lfloor d/2 \bigr\rfloor \bmod 2, \quad r_y = (r_x \oplus d) \bmod 2$$
with a conditional 90° rotation applied at each level before accumulating the grid offset.

**Logic in the Code**
1. `_hilbert_xy` loops over all $4^k$ indices; for each it iterates over $k$ bit-pair levels, applying the rotation and offset to accumulate $(x, y)$.
2. The resulting $(4^k, 2)$ coordinate array is stacked into $(4^k{-}1, 2, 2)$ segments for a `LineCollection` draw — no individual `ax.plot` calls.
3. The palette gradient maps position along the 1D index to colour, so the progression of the space-filling traversal is immediately visible.

**Interesting Properties**
The Hilbert Curve's locality preservation is quantified: the Euclidean distance between two grid cells never exceeds $O(\sqrt{\Delta d})$ where $\Delta d$ is their 1D index distance — far superior to the $O(n)$ worst case of a row-scan. Higher orders reveal the same rotated U-shape repeating at every scale, a canonical example of exact geometric self-similarity.

### How This Works — L-System Tree

**Core Idea**
An L-System (Lindenmayer System) is a formal string-rewriting grammar that applies production rules to every symbol in parallel at each step. When the resulting string is interpreted as turtle-graphics commands, it produces branching structures that closely resemble real plants. Aristid Lindenmayer introduced them in 1968 to model algae cell growth.

**Mathematics**
The system uses axiom $\omega = X$ with two simultaneous rules:
$$X \;\to\; F{+}[[X]{-}X]{-}F[{-}FX]{+}X \qquad F \;\to\; FF$$
After $n$ iterations the number of $F$ symbols (drawable segments) grows as $\approx 896$ at $n=7$ and the string length as $O(2^n)$. The turtle interprets: $F$ = step forward; $+/-$ = turn $\pm\delta$ (the *Angle* control); $[$ = push state; $]$ = pop state. The branching angle $\delta$ is the dominant shape parameter: small $\delta$ yields narrow spruce forms; large $\delta$ yields spreading oaks.

**Logic in the Code**
1. The axiom is expanded $n$ times using a single `str.join` pass per iteration — substituting every character via a dictionary lookup.
2. A turtle state $(x, y, \theta, \text{depth})$ is maintained with a stack. Each `F` draws a segment and records the current stack depth.
3. Segments are coloured by depth via the palette so trunk segments (depth 0) and fine twigs (deepest depth) receive distinct hues, giving a natural light-to-shadow gradient.
4. All segments are rendered in one `LineCollection` call.

**Interesting Properties**
This rule set is Prusinkiewicz and Lindenmayer's "branching tree" from *The Algorithmic Beauty of Plants* (1990). Changing only the angle from 25° to 30° dramatically shifts the silhouette from a symmetric fern to a wind-swept deciduous shape, illustrating how a single parameter controls emergent biological form.

### How This Works — Apollonius Gasket

**Core Idea**
The Apollonius Gasket is a fractal constructed by repeatedly packing circles into the curved triangular gaps formed by three mutually tangent circles. Starting from three equal circles inside a bounding circle, each gap is filled with a new tangent circle, then the three new gaps around that circle are filled, and so on infinitely. The result is a fractal arrangement of tangent circles whose boundary has Hausdorff dimension ≈ 1.305.

**Mathematics**
The key tool is **Descartes' Circle Theorem** (1643): if four circles are mutually tangent with curvatures $k_1, k_2, k_3, k_4$ (where $k = 1/r$ and $k < 0$ for the outer bounding circle), then:
$$(k_1+k_2+k_3+k_4)^2 = 2(k_1^2+k_2^2+k_3^2+k_4^2)$$
Solving for the unknown curvature gives:
$$k_4 = k_1+k_2+k_3 \pm 2\sqrt{k_1 k_2 + k_2 k_3 + k_1 k_3}$$
The **Extended Descartes theorem** adds center positions (as complex numbers $z = x+iy$):
$$k_4 z_4 = k_1 z_1 + k_2 z_2 + k_3 z_3 \pm 2\sqrt{k_1 k_2 z_1 z_2 + k_2 k_3 z_2 z_3 + k_1 k_3 z_1 z_3}$$
The $\pm$ sign in both formulas is the **same** — the two solutions correspond to the two circles tangent to a given triple. The three equal inner circles have curvature $k_s = 1 + 2/\sqrt{3}$ (from Descartes applied to the initial quadruple $k_0{=}{-}1, k_1{=}k_2{=}k_3{=}k_s$).

**Logic in the Code**
1. The gasket is built via BFS. The initial four circles seed a queue with 4 "gap triples": $(c_0,c_1,c_2)$, $(c_0,c_1,c_3)$, $(c_0,c_2,c_3)$, $(c_1,c_2,c_3)$.
2. For each triple, both Descartes solutions are computed. Circles with negative/zero curvature or outside the bounding disk are rejected.
3. Each new circle $c_4$ spawns 3 new triples — the 3 new gaps it creates with the parent triple. A rounding-based key prevents duplicates.
4. BFS stops when the exploration depth or a circle-count cap is reached. Circles are drawn largest-first and colored by depth level.

**Interesting Properties**
The Apollonius Gasket is a classic example of a **fractal limit set**. The total area of all circles converges to the area of the bounding circle, but the boundary between filled and unfilled regions is a fractal curve. Every tangent point between two circles is a point of the fractal limit set — the set of points never covered by any circle, which has measure zero but Hausdorff dimension ≈ 1.305.

### How This Works — Lissajous Figures

**Core Idea**
A Lissajous figure is the trajectory traced by a point whose $x$ and $y$ coordinates oscillate sinusoidally at (generally different) frequencies. The resulting closed curves — named after Jules Antoine Lissajous, who demonstrated them optically in 1857 — appear in oscilloscope displays, mechanical systems, and signal processing.

**Mathematics**
The parametric equations are:
$$x(t) = \sin(a\,t + \delta), \qquad y(t) = \sin(b\,t)$$
where $a$ and $b$ are the frequency integers and $\delta$ is the phase difference. The curve is **closed** (periodic) whenever $a/b$ is rational — which it always is for integer $a, b$. The full period is:
$$T = \frac{2\pi}{\gcd(a,b)}$$
The shape depends critically on the phase $\delta$: at $\delta = 0$ the curve is a straight line (or an oblique ellipse for $a \neq b$); at $\delta = \pi/2$ a symmetric, "knotted" figure emerges. The number of lobes equals $a$ in one direction and $b$ in the other.

**Logic in the Code**
1. A time vector $t \in [0, 2\pi/\gcd(a,b))$ is sampled uniformly with `n_pts` points — exactly one full period, ensuring the curve closes perfectly.
2. $x(t)$ and $y(t)$ are computed via NumPy broadcasting (no Python loop over samples).
3. Consecutive coordinate pairs are stacked into a `(N-1, 2, 2)` segment array and rendered by `LineCollection`, coloring each segment by its position in $t$ to trace the trajectory's progression.

**Interesting Properties**
When $a/b$ is irrational (approximated with large integers), the curve never closes and densely fills an ellipse over infinite time. Setting $a = b$ and varying $\delta$ traces a circle ($\delta = \pi/2$) → ellipse → line ($\delta = 0$) — the entire family of $\delta$-parameterized curves is used in phase-comparison oscilloscopes. The Lissajous family with $a = b \pm 1$ produce the familiar "figure-8" shapes.

### How This Works — Rose Curves

**Core Idea**
Rose curves (rhodonea curves) are polar curves whose petals emerge from a simple cosine equation. The ratio of two integers determines how many petals appear and how the curve winds around the origin. First studied by Luigi Guido Grandi in 1728, they are among the most elegant examples of how a single parameter change can radically transform a geometric shape.

**Mathematics**
The polar equation is:
$$r = \cos\!\left(\tfrac{p}{q}\,\theta\right)$$
where $p$ and $q$ are coprime integers (reduced to lowest terms). To guarantee closure the parameter must sweep:
$$\theta \in \left[0,\; 2\pi q\right]$$
Petal count depends on the parity of $p$ and $q$: integer $k = p/q$ gives $k$ petals when $k$ is odd and $2k$ when $k$ is even. For non-integer rational $p/q$ the count formula is more complex, but the curves always close after exactly $q$ full revolutions and produce striking layered multi-petal shapes.

**Logic in the Code**
1. $p$ and $q$ are reduced by their GCD so sliders always produce valid coprime ratios.
2. A uniform $\theta$ grid over $[0, 2\pi q]$ is generated; $r = \cos(p/q \cdot \theta)$ is evaluated; Cartesian coordinates follow from $x = r\cos\theta$, $y = r\sin\theta$. Negative $r$ values fold back, tracing petals on the opposite side.
3. A `LineCollection` colors each tiny arc segment by its position in $\theta$, creating a rainbow gradient that reveals the winding order of the petals.

**Interesting Properties**
Rose curves are a subset of the broader family of **harmonograph** curves and share the same parametric structure as Lissajous figures (with $y$ replaced by a radial coordinate). Irrational ratios $p/q$ (approximated by large integers) produce curves with an astronomically large period that appear to fill a disk with dense overlapping petals before eventually closing — a finite analogue of an irrational orbit on a torus.


### How This Works — Chaos Attractor (Lorenz)

**Core Idea**
The Lorenz system is a simplified model of atmospheric convection derived by Edward Lorenz in 1963. Despite having only three variables and three parameters, it exhibits **sensitive dependence on initial conditions** — the butterfly effect — producing a non-repeating trajectory that winds forever around two lobes of a strange attractor without ever crossing itself.

**Mathematics**
The system of ODEs is:
$$\frac{dx}{dt} = \sigma(y - x), \qquad \frac{dy}{dt} = x(\rho - z) - y, \qquad \frac{dz}{dt} = xy - \beta z$$
The classical parameters $\sigma = 10$, $\rho = 28$, $\beta = 8/3$ produce chaotic dynamics. The attractor lives in a bounded region of 3D space; the $x$–$z$ projection — the "butterfly wings" view — is the most recognizable cross-section. Lyapunov exponent $\lambda_1 \approx 0.91$ quantifies the exponential divergence of nearby trajectories.

**Logic in the Code**
1. The trajectory is integrated via classical **4th-order Runge–Kutta (RK4)** using Python floats in a tight loop for speed (no per-step NumPy overhead).
2. All three stages of each RK4 step are inlined as explicit scalar operations; the 3 arrays `traj_x/y/z` are pre-allocated and filled element by element.
3. The first `skip` steps (transient) are discarded; the remaining $x$–$z$ coordinates form a `(N, 2)` path that is split into `LineCollection` segments colored by time, so the early orbit (cooler hues) and later orbit (warmer hues) are visually distinct.
4. The `σ`, `ρ`, `β` sliders let users explore parameter space: reducing `ρ` below ~24.74 destroys chaos and the trajectory converges to a fixed point; raising it re-enters chaos.

**Interesting Properties**
The Lorenz attractor has a fractal cross-section: slicing it with a plane perpendicular to the orbit reveals a Cantor-like set rather than a smooth curve. Its Hausdorff dimension is approximately 2.06 — just barely more than a surface. Lorenz's 1963 discovery launched the modern study of chaos theory and nonlinear dynamics.


### How This Works — Wave Interference Pattern

**Core Idea**
When two or more coherent waves from point sources overlap, they add algebraically — **constructive interference** where crests meet (bright bands) and **destructive interference** where a crest meets a trough (dark bands). The result is a standing fringe pattern that encodes the geometry of the source arrangement. This principle underlies double-slit experiments, radio antenna arrays, and optical holography.

**Mathematics**
Each source $i$ emits a circular wave. At a field point $(x, y)$, the amplitude contributed by source $i$ is:
$$A_i(x,y) = \cos(k\, d_i), \qquad d_i = \sqrt{(x-x_i)^2+(y-y_i)^2}$$
where $k = 2\pi/\lambda$ is the wavenumber. The total field is the **superposition**:
$$A(x,y) = \sum_{i=1}^{n} A_i(x,y)$$
This assumes equal-amplitude, equal-phase, monochromatic sources (fully coherent). The fringe spacing near the centre scales as $\lambda / \theta$ where $\theta$ is the angle subtended by the source pair, matching the classical Young's fringe formula.

**Logic in the Code**
1. $n$ source positions are drawn from a seeded RNG inside a circle of radius 70 units (coordinate system spans ±100), so changing the `Seed` slider gives a new random arrangement.
2. A $\text{res} \times \text{res}$ pixel grid is generated once via `np.meshgrid`; for each source, `np.hypot` computes all distances in a single vectorised call. The `np.maximum(d, 0.5)` guard prevents a numerical singularity at the source location.
3. The summed field is normalised to $[-1, 1]$ and rendered with `imshow` using the selected colormap (diverging palettes like *Arctic Aurora* show constructive maxima and destructive minima most clearly). Source positions are overlaid as white `+` markers.
4. The `Wavelength` slider (in grid units spanning ±100) directly controls $\lambda$; smaller values produce denser fringes with more visible concentric rings per source.

**Interesting Properties**
With two sources separated by distance $d$, the path-length difference to any far-field point is $d\sin\theta$; bright fringes occur when this equals $m\lambda$ (integer multiples). Increasing the number of sources from 2 to $n$ narrows the principal maxima and introduces $n-2$ secondary maxima between them — the same principle used in phased-array radar and Wi-Fi beamforming to concentrate transmitted power in a specific direction.

### How This Works — Hypocycloid & Epicycloid

**Core Idea**
A hypocycloid is the curve traced by a point fixed to a small circle that rolls *inside* a larger fixed circle. An epicycloid traces a point on a circle rolling *outside* the fixed circle. By varying the ratio of radii and the pen distance, an enormous family of closed, star-like, and petal curves emerges — from the simple cardioid (epicycloid, r = R) to the sharp-pointed astroid (hypocycloid, r = R/4).

**Mathematics**
For a fixed circle of radius $R$ and a rolling circle of radius $r$ with the pen at distance $d$ from the rolling circle's center:

*Hypocycloid (hypotrochoid):*
$$x(t) = (R-r)\cos t + d\cos\!\left(\frac{R-r}{r}\,t\right), \qquad y(t) = (R-r)\sin t - d\sin\!\left(\frac{R-r}{r}\,t\right)$$

*Epicycloid (epitrochoid):*
$$x(t) = (R+r)\cos t - d\cos\!\left(\frac{R+r}{r}\,t\right), \qquad y(t) = (R+r)\sin t - d\sin\!\left(\frac{R+r}{r}\,t\right)$$

The curve closes after $T = 2\pi \cdot R / \gcd(R, r)$ radians of $t$ (using integer approximations of $R$ and $r$). When $d = r$ the pen rides on the rim and the generic curve specializes to a true hypo- or epicycloid; for $d \neq r$ it is a *trochoid*.

**Logic in the Code**
1. Integer approximations of $R$ and $r$ compute the period via `gcd` so the curve closes cleanly.
2. `np.linspace` generates $n$ evenly-spaced $t$ values over the full period; both $x$ and $y$ are evaluated in a single NumPy broadcast.
3. Consecutive coordinate pairs form a `LineCollection` with a palette gradient mapped along $t$, so the winding order of the curve is visible as a color progression.
4. The **Type** dropdown switches between hypo- and epi- formulas; all three sliders ($R$, $r$, $d$) are live and immediately change curve topology.

**Interesting Properties**
Special cases: $r = R/4$ → astroid (4-pointed star); $r = R/3$ → deltoid (3-pointed); epi with $r = R$ → cardioid. When $R/r$ is irrational the curve never closes and densely fills an annulus — analogous to an irrational rotation on a torus. The Spirograph toy marketed from 1965 onward is a physical implementation of the hypotrochoid family.

---

### How This Works — Truchet Tiles

**Core Idea**
In 1704, Sébastien Truchet described a simple square tile with a diagonal stripe and observed that randomly orienting such tiles across a grid produces surprisingly rich, organic-looking patterns — including winding rivers, maze corridors, and Celtic knotwork motifs. The arc variant (two quarter-circle arcs per tile) is the modern standard and generates flowing, coral-like networks.

**Mathematics**
Each tile occupies a unit square $[0,1]^2$. In the arc variant two quarter-circles of radius $\tfrac{1}{2}$ connect the midpoints of adjacent sides:
- **Orientation 0:** arc centred at $(0, 0)$ connecting the bottom and left midpoints; arc centred at $(1, 1)$ connecting the top and right midpoints.
- **Orientation 1:** arc centred at $(1, 0)$ and arc centred at $(0, 1)$.

A uniformly random binary matrix (one bit per tile) selects the orientation. For an $n \times n$ grid there are $2^{n^2}$ possible configurations, but all typical realizations share the same visual statistics.

**Logic in the Code**
1. `np.random.default_rng(seed).integers(0, 2, (n, n))` generates the orientation matrix in one call.
2. For each cell, the correct pair of `matplotlib.patches.Arc` objects (or diagonal line segments for the "Diagonals" style) is added to the axes. Color is sampled from the palette gradient indexed by row, giving a horizontal rainbow banding effect.
3. The "Mixed" style uses arcs in even-parity cells ($(row + col) \bmod 2 = 0$) and diagonals in odd-parity cells, producing a hybrid texture.
4. All arcs are drawn with `ax.add_patch` — no rasterization — so the output is fully vector and exports cleanly at any DPI.

**Interesting Properties**
At the percolation threshold (50% probability per orientation), connected arc-paths form a fractal cluster that spans the grid. Smith, Dimond, and Truchet (independently) showed that the expected length of a connected path scales as $O(n^{4/3})$ — the same universality class as 2D percolation. The "Smith chart" used in RF engineering is a distant relative of this tile family.

---

### How This Works — Hexagonal Grid Art

**Core Idea**
The regular hexagonal lattice is the unique tiling of the plane that maximizes the ratio of enclosed area to perimeter — the same reason honeycombs use hexagons. By assigning colors to each cell based on its axial coordinates, a wide variety of interference patterns, gradients, and geometric motifs emerge from the same underlying tessellation.

**Mathematics**
Hexagonal grids are naturally described in *axial coordinates* $(q, s)$ where $r = -q - s$. For a pointy-top layout with cell size $a$, the Cartesian center of cell $(q, s)$ is:
$$x = a\sqrt{3}\!\left(q + \tfrac{s}{2}\right), \qquad y = a \cdot \tfrac{3}{2} \cdot s$$
The six corners of a hexagon at center $(c_x, c_y)$ are:
$$\left(c_x + a\cos\!\left(\frac{\pi k}{3} + \frac{\pi}{6}\right),\; c_y + a\sin\!\left(\frac{\pi k}{3} + \frac{\pi}{6}\right)\right), \quad k = 0,\ldots,5$$
A grid of $r$ rings contains $(3r^2 + 3r + 1)$ hexagons — 127 cells at $r = 6$, 1141 at $r = 18$.

**Logic in the Code**
1. The double loop over $(q, s) \in [-\text{rings}, \text{rings}]^2$ keeps only cells where $|{-q-s}| \leq \text{rings}$, correctly forming a hexagonal outline.
2. Four coloring modes map each cell to a value $t \in [0,1]$: **Distance** (Euclidean distance from origin), **Angle** (azimuth), **Checkerboard** ($(q+s+r) \bmod 3$, producing a 3-color tiling), **Random** (seeded RNG).
3. Each $t$ value indexes the selected colormap; cells are drawn as filled `Polygon` patches with a thin semi-transparent edge to show the grid structure.
4. The `Hex Size` slider scales all coordinates uniformly; combined with `Rings`, it controls the visual density independently of window size.

**Interesting Properties**
The axial coordinate system makes neighbor queries trivial: the six neighbors of $(q, s)$ are exactly $(q \pm 1, s)$, $(q, s \pm 1)$, $(q+1, s-1)$, $(q-1, s+1)$ — no trigonometry required. The checkerboard coloring with 3 colors is the unique proper 3-coloring of the hexagonal lattice, isomorphic to the three-state Potts model studied in statistical mechanics.

---

### How This Works — Spirograph Generator

**Core Idea**
The Spirograph toy (Kenner, 1965) uses physical gears to draw hypotrochoid curves. The pen rides inside a hole in a small plastic gear rolling around the inside of a fixed ring gear. By stacking multiple curves with increasing pen distances, the generator creates the layered, overlapping petal patterns seen in commercial Spirograph art packs.

**Mathematics**
Each layer draws a hypotrochoid with fixed $R$ (outer teeth) and $r$ (inner teeth) but a different pen distance $d_\ell$:
$$d_\ell = r \cdot \left(0.3 + 0.7 \cdot p \cdot \frac{\ell+1}{L}\right), \quad \ell = 0, \ldots, L{-}1$$
where $p$ is the **Pen Ratio** slider and $L$ is the number of layers. This linearly spaces the pen from $0.3r$ (close to the gear center, tight loops) to $p \cdot r$ (near the rim, wide loops).

The curve closes after $T = 2\pi \cdot R / \gcd(R, r)$ radians; integer tooth counts guarantee rational $R/r$ and exact closure.

**Logic in the Code**
1. Teeth counts are converted to floating-point radii; `gcd` gives the exact period.
2. A loop over `layers` computes each hypotrochoid independently using vectorized NumPy; consecutive pairs become `LineCollection` segments.
3. Each layer receives a distinct color from the palette gradient, so inner and outer layers have contrasting hues and the overlap region blends visually.
4. Alpha is set to 0.80 per layer so overlapping regions darken naturally, mimicking the ink accumulation of a real physical spirograph.

**Interesting Properties**
When $\gcd(R, r) = 1$ (coprime teeth), the curve must complete $R$ full rotations of the inner gear before closing — producing the maximum number of petals ($R$). Composite $R/r$ ratios close sooner and produce fewer, wider petals. The limiting case $r \to R$ (gear fills the ring) traces a straight line — the degenerate Tusi couple used in medieval Islamic astronomy to convert circular motion to linear.

---

### How This Works — Parametric Curve Art

**Core Idea**
Parametric curves define position as a pair of functions $(x(t), y(t))$ of a single parameter $t$. Unlike Cartesian equations $y = f(x)$, they naturally represent multi-valued, self-intersecting, and looping paths. This pattern is a curated gallery of eight famous named curves, each with distinct mathematical origins but all renderable by the same `LineCollection` pipeline.

**Mathematics — Selected Curves**

| Curve | Equations | Origin |
|---|---|---|
| **Butterfly** | $x = \sin t \cdot e(t)$, $y = \cos t \cdot e(t)$, $e = e^{\cos t} - 2\cos 4t - \sin^5(t/12)$ | Temple H. Fay, 1989 |
| **Astroid** | $x = \cos^3 t$, $y = \sin^3 t$ | Roemer & Bernoulli, 1674 |
| **Superellipse** | $x = \text{sgn}(\cos t)|\cos t|^{2/n}$, $y = \text{sgn}(\sin t)|\sin t|^{2/n}$, $n=2.5$ | Gabriel Lamé, 1818 |
| **Trisectrix** | $r = 1 + 2\cos\theta$ (polar) | MacLaurin, 1742 |
| **Folium of Descartes** | $x = 3a\tan t/(1+\tan^3 t)$, $y = 3a\tan^2 t/(1+\tan^3 t)$ | Descartes, 1638 |

The Epitrochoid Star ($R=5$, $r=3$, $d=5$) and Hypotrochoid Flower ($R=7$, $r=3$, $d=6$) are trochoid special cases already described under Pattern 16.

**Logic in the Code**
1. Each curve name maps to a `(t_min, t_max)` range chosen so the curve completes exactly one visual cycle without excess overlap.
2. `_compute_curve` evaluates the parametric equations as NumPy arrays — fully vectorized, no Python loop over points.
3. A finite-value mask (`np.isfinite`) removes any NaN/Inf values that arise at algebraic singularities (notably the Folium's asymptote at $\tan t = -1$).
4. The resulting segments are colored by $t$-position along the gradient, revealing the temporal winding order of the curve — where it starts, how it self-intersects, and where it ends.

**Interesting Properties**
The Butterfly curve ($e^{\cos\theta} - 2\cos 4\theta - \sin^5(\theta/12)$) was discovered empirically by Temple Fay while exploring polar-to-parametric transformations; it has no known closed-form area or arc-length. The Superellipse with $n = 2$ is a circle; $n < 2$ gives a "squircle" (between square and circle); the limit $n \to \infty$ approaches a perfect square. Danish designer Piet Hein used superellipses ($n = 2.5$) to design Sergels Torg plaza in Stockholm in 1959.

### How This Works — Cherry Blossom Particle Scene

**Core Idea**
A cherry blossom scene combines two elements: a procedural tree built by recursive binary branching, and a particle system of petals scattered across the canvas. Wind displaces petals horizontally in proportion to their height — petals near the top have fallen farther and drifted more. The result captures the ephemeral, gravity-driven cascade of *hanami* (flower viewing).

**Mathematics**
The recursive branching follows the same geometric principle as a binary fractal tree. Each branch of length $\ell$ at angle $\theta$ spawns two children at angles $\theta \pm \delta$ with length $\ell \cdot r$ where $r \approx 0.67$ is the decay ratio. With depth $d$, the maximum number of segments is $2^d - 1$ (plus occasional third branches for bushy variation).

Petal horizontal displacement due to wind is modeled as:
$$x' = x + w \cdot (1 - y) \cdot \varepsilon, \qquad \varepsilon \sim \mathcal{U}(-0.3, 0.3)$$
where $w$ is the Wind control and $(1 - y)$ ensures petals near the top (large fall distance) drift more than those near the ground.

**Logic in the Code**
1. The `_branch` function accumulates all segments into a list before drawing. A single `LineCollection` draws all branch segments with per-segment colors and linewidths in one call — avoiding thousands of individual `ax.plot` calls.
2. Petal positions are computed via fully vectorized NumPy broadcasting: `n_petals` random $(x, y)$ points, shifted by the wind formula, then clipped to canvas bounds.
3. Petal RGB colours are linearly interpolated from deep rose $(1, 0.35, 0.50)$ to pale blush $(1, 0.85, 0.90)$ using a per-petal hue parameter drawn from $\mathcal{U}(0,1)$.
4. A separate ground scatter plots smaller, semi-transparent petals in the bottom 5.5% of the canvas to simulate accumulation.

**Interesting Properties**
The "third branch" added with 35% probability when depth $> 3$ breaks perfect binary symmetry, producing the irregular silhouette characteristic of real *Prunus serrulata* trees. Increasing wind past 0.7 causes petals to cluster on the leeward side — a qualitatively correct simulation of how wind-driven cherry blossom showers behave in nature.

### How This Works — Procedural Tree Generator

**Core Idea**
A procedural tree is generated by direct recursive geometric branching — no string rewriting, no turtle grammar. At each node a trunk segment gives birth to $n$ child branches, each rotated by a fraction of the configured angle. Small random jitter on each branch's angle and length makes the result organic rather than mechanical. This approach is faster than L-Systems for interactive use and allows more direct control over tree shape.

**Mathematics**
Given a parent branch of length $\ell$ at heading $\theta$ and branching count $n$, the child headings are evenly spaced in the range $[\theta - \phi/2,\;\theta + \phi/2]$:
$$\theta_i = \theta - \phi/2 + \phi \cdot \frac{i}{n-1}, \qquad i = 0, \ldots, n-1$$
where $\phi$ is the Branch Angle parameter. Each child has length:
$$\ell' = \ell \cdot (r + \varepsilon), \qquad \varepsilon \sim \mathcal{U}(-0.03, 0.03)$$
where $r$ is the Length Decay slider. Total segment count is bounded by $n^d$ where $d$ is the depth.

**Logic in the Code**
1. All segments are accumulated in lists (`seg_list`, `t_list`, `lw_list`) without drawing. The recursion only builds data structures.
2. A single `LineCollection` draws all segments simultaneously with per-segment colors (from the selected palette mapped to tree depth $t = 1 - d/\text{depth}$) and per-segment linewidths (thicker near the trunk). This replaces tens of thousands of individual `ax.plot` calls.
3. The palette gradient runs from trunk (near palette start) to tip (near palette end), so the colour encodes tree anatomy: darker base, brighter growing tips.
4. The `_draw` function is bounded to avoid runaway recursion: it exits when segment length drops below 0.005 canvas units, regardless of depth.

**Interesting Properties**
With `n_branches = 2` and `branch_angle ≈ 25°`, the tree resembles a *Populus* poplar. Increasing to `n_branches = 3` and `branch_angle ≈ 40°` produces oak-like spreading crowns. At `length_decay = 0.5` the tree converges rapidly and stays compact; at `0.85` it spreads into a sprawling canopy — illustrating how a single allometric coefficient shapes an entire plant architecture.

### How This Works — Reaction-Diffusion (Turing Patterns)

**Core Idea**
Reaction-diffusion systems describe how chemical species spread through space and react with each other. Alan Turing showed in 1952 that two interacting chemicals — an *activator* and an *inhibitor* — can spontaneously break spatial symmetry and produce the spots, stripes, and labyrinthine patterns seen on animal skins, fish fins, and seashells. The Gray-Scott model is the canonical computational formulation.

**Mathematics**
The Gray-Scott model tracks two concentrations, $A$ (substrate) and $B$ (product):
$$\frac{\partial A}{\partial t} = D_A \nabla^2 A - AB^2 + f(1 - A)$$
$$\frac{\partial B}{\partial t} = D_B \nabla^2 B + AB^2 - (f + k)B$$
The reaction $A + 2B \to 3B$ converts the substrate autocatalytically: one molecule of $B$ recruits two more, turning $A$ into $B$. The feed parameter $f$ replenishes $A$ from a reservoir; the kill parameter $k$ removes $B$. The diffusion constants $D_A = 1.0$ and $D_B = 0.5$ ensure the activator ($B$) diffuses more slowly than the substrate — a necessary condition for pattern formation. Different $(f, k)$ coordinates produce qualitatively different morphologies: spots at $(0.037, 0.060)$, worms at $(0.025, 0.060)$, mazes at $(0.029, 0.057)$.

**Logic in the Code**
1. The grid is initialized with $A = 1$ and $B = 0$ everywhere, then a set of small random square patches are seeded with $A = 0.5$, $B = 0.25$ to break spatial symmetry.
2. The discrete Laplacian is computed at each step via four `np.roll` calls (one per compass direction): $\nabla^2 A \approx A_{i-1,j} + A_{i+1,j} + A_{i,j-1} + A_{i,j+1} - 4A_{i,j}$, with periodic boundary conditions.
3. The reaction and diffusion terms are computed fully vectorized across the $N \times N$ array; `np.clip` keeps both fields in $[0, 1]$ after each step.
4. After all iterations, the $B$ field is rendered via `imshow`; the palette colormap maps concentration to colour.

**Interesting Properties**
The $(f, k)$ parameter space has a rich "phase diagram": crossing the Turing instability boundary flips the pattern from uniform equilibrium to spatial structure. The same Gray-Scott equations that produce animal skin patterns also describe electrode corrosion, chemical oscillators (BZ reaction), and some bacterial colony growth patterns.

### How This Works — Flocking Birds (Boids Lite)

**Core Idea**
Boids (Bird-oid Objects) is Craig Reynolds' 1986 model showing that coherent flocking behavior emerges from three simple local rules applied independently by each agent — no central controller, no global information. The resulting collective motion closely matches the murmurations of starlings, schools of fish, and swarms of insects.

**Mathematics**
Each boid $i$ at position $\mathbf{p}_i$ with velocity $\mathbf{v}_i$ applies three steering forces at each time step:

**Separation** — avoid crowding neighbors within radius $r_s$:
$$\mathbf{f}_s = \sum_{j \in \mathcal{N}_s(i)} (\mathbf{p}_i - \mathbf{p}_j), \quad \text{normalized}$$

**Alignment** — steer toward average heading of neighbors within radius $r_a$:
$$\mathbf{f}_a = \frac{1}{|\mathcal{N}_a|} \sum_{j \in \mathcal{N}_a(i)} \mathbf{v}_j, \quad \text{normalized}$$

**Cohesion** — steer toward center of mass of neighbors within radius $r_c$:
$$\mathbf{f}_c = \frac{1}{|\mathcal{N}_c|} \sum_{j \in \mathcal{N}_c(i)} \mathbf{p}_j - \mathbf{p}_i, \quad \text{normalized}$$

The combined update is $\mathbf{v}_i \mathrel{+}= (w_s \mathbf{f}_s + w_a \mathbf{f}_a + w_c \mathbf{f}_c) \cdot v_{\max}$, then speed is clamped to $v_{\max}$.

**Logic in the Code**
1. Pairwise differences are computed as a single $(N, N, 2)$ tensor: `diff = pos[:, None] - pos[None]`. Minimum-image periodic wrapping (`diff -= np.round(diff)`) makes the space toroidal.
2. Neighbor masks are Boolean $(N, N, 1)$ broadcasts applied to `diff` and `vel[None]` in one NumPy multiply-sum, avoiding all Python loops over boids.
3. Each force vector is normalized to unit length before scaling by its weight and $v_{\max}$, so the weight sliders have consistent effect regardless of local crowd density.
4. The final snapshot is rendered as a `quiver` plot: arrow direction = normalised velocity, arrow color = speed (mapped through the selected palette).

**Interesting Properties**
The three radii satisfy $r_s < r_a < r_c$ — a necessary ordering. If $r_s \geq r_a$ boids cannot align without first repelling each other, causing the flock to disintegrate. Setting $w_s \gg w_a, w_c$ produces an individual-avoidance swarm with no coherent direction; setting $w_c \gg$ others collapses the flock into a single spinning clump.

### How This Works — Lightning Bolt Generator

**Core Idea**
Lightning is a plasma discharge that travels from cloud to ground via the path of least electrical resistance. In reality this path is determined by the probabilistic dielectric breakdown of air — each step is biased toward lower potential but includes random components. The midpoint displacement algorithm captures this stochastic branching structure: a straight path is iteratively perturbed and split into zigzag channels, with random branches splitting off at each level.

**Mathematics**
Given a segment from $\mathbf{p}_1$ to $\mathbf{p}_2$, the midpoint is displaced perpendicular to the segment:
$$\mathbf{m} = \frac{\mathbf{p}_1 + \mathbf{p}_2}{2} + \lambda \cdot \|\mathbf{p}_2 - \mathbf{p}_1\| \cdot \hat{\mathbf{n}} \cdot \varepsilon$$
where $\hat{\mathbf{n}}$ is the unit normal to the segment, $\varepsilon \sim \mathcal{U}(-1, 1)$, and $\lambda$ is the Roughness parameter. Recursing to depth $d$ halves the segment length each time; the total number of base segments is $2^d$. Branches are spawned at each recursion level with probability $p_{\text{branch}}$ — they are shorter-lived (reaching only depth $d - 2$) so they taper convincingly.

**Logic in the Code**
1. The main `_bolt(p1, p2, d, intensity)` function recurses exactly twice per call (left and right half-segments), building a binary tree of segments at the base case ($d = 0$).
2. At each level, with probability `branch_prob` a side channel is launched from the current midpoint toward a randomly deflected endpoint. Its intensity is scaled by 0.45 so branches render thinner and dimmer than the main bolt.
3. All accumulated segments are drawn after recursion completes: a narrow bright core `(0.7+0.3i, 0.8+0.2i, 1.0)` with alpha ∝ intensity, plus a wide low-alpha glow pass in blue to simulate plasma emission.
4. The Roughness slider directly scales $\lambda$: near zero gives a nearly straight bolt; near 0.8 gives chaotic, fractal zigzags typical of megavolt discharges.

**Interesting Properties**
The midpoint displacement algorithm is the same technique used to generate fractal terrain (Diamond-Square) — the same mathematical structure underlies both lightning and mountain ridges. Real lightning exhibits a Hausdorff dimension of approximately 1.7, similar to the dimension of DLA (Diffusion-Limited Aggregation) clusters, because both processes are governed by Laplacian growth.

### How This Works — Snowflake Crystal Growth

**Core Idea**
Real snowflakes grow by water vapour depositing onto an ice crystal. Because hexagonal ice has six-fold symmetry and the local temperature and humidity are nearly identical across the tiny crystal, all six arms grow almost identically — producing the famous six-fold symmetric, yet uniquely detailed, snowflake forms. The simulation captures this by explicitly enforcing six-fold symmetry in a recursive arm-and-branch structure.

**Mathematics**
Each of the six arms grows from the center at angle $k\pi/3$, $k = 0, \ldots, 5$. Along each arm, the main spine recurses with scale factor $r_{\text{arm}}$ (the Arm Scale slider):
$$\ell_{d} = \ell_0 \cdot r_{\text{arm}}^{\,d}$$
At each spine node, two side branches grow at angles $\pm(\pi/3 + \delta_d)$ from the spine direction, where $\delta_d$ is a small seeded jitter providing organic variation without breaking overall six-fold symmetry. Side branches use scale factor $r_{\text{branch}}$ (the Branch Scale slider):
$$\ell_{\text{branch}} = \ell_{\text{spine}} \cdot r_{\text{branch}}$$
Side branches also recurse, producing sub-branches — giving the characteristic hierarchical complexity.

**Logic in the Code**
1. The `_arm(x, y, angle, length, d)` function accumulates all segment data (`(x1,y1), (x2,y2), t`) into lists without drawing. This separates geometry computation from rendering.
2. After all six arms are traced, a single `LineCollection` renders all segments with per-segment colors (palette mapped to depth fraction $t = 1 - d/(depth+1)$) and per-segment linewidths (trunk thick, tips thin).
3. A second `LineCollection` with $5\times$ wider linewidth and $\approx 6\%$ alpha draws a soft glow halo over the crystal, simulating refracted light.
4. The per-depth jitter array `branch_off` is pre-computed from the seeded RNG before recursion begins, guaranteeing that all six arms receive identical jitter (correct six-fold symmetry) while still looking organic.

**Interesting Properties**
Real snowflake diversity arises because each crystal's history of temperature and humidity varies along its growth path — so two crystals growing a millimetre apart can look completely different, even though both are perfectly six-fold symmetric. At the Branch Scale boundary $r_{\text{branch}} = 1.0$, side branches grow as long as the spine — producing a fractal where every sub-branch looks like the whole snowflake (exact self-similarity).

### How This Works — Leaf Venation Simulation

**Core Idea**
Leaf veins form through a process called auxin canalization: the plant hormone auxin flows from the leaf margin inward, and wherever auxin concentration is high, the cell wall stiffens and becomes a conducting channel. The space colonization algorithm by Runions et al. (2007) models this by placing attractor points (representing auxin sources) throughout the leaf and growing vein segments toward whichever attractors are nearest. Once a vein reaches an attractor's kill radius, that source is consumed and vein growth pivots toward the next cluster.

**Mathematics**
Let $\mathbf{A} = \{a_1, \ldots, a_M\}$ be the attractor points and $\mathbf{N} = \{n_1, \ldots, n_K\}$ the current vein node positions. At each iteration:

1. For each attractor $a_i$, find its closest node: $n^*(i) = \arg\min_j \|a_i - n_j\|$.
2. Remove attractors within the kill radius: $\{a_i : \|a_i - n^*(i)\| \leq r_k\}$.
3. For each active node $n_j$ that attracted at least one source, compute the normalized mean direction:
$$\mathbf{d}_j = \frac{\displaystyle\sum_{i : n^*(i) = j} \frac{a_i - n_j}{\|a_i - n_j\|}}{\left\|\displaystyle\sum_{i : n^*(i) = j} \frac{a_i - n_j}{\|a_i - n_j\|}\right\|}$$
4. Grow a new node: $n_{\text{new}} = n_j + s \cdot \mathbf{d}_j$ where $s$ is the step size, if $n_{\text{new}}$ lies inside the leaf ellipse.

**Logic in the Code**
1. Attractors are sampled uniformly inside the ellipse using rejection sampling. The root node starts at the leaf base (bottom of the ellipse).
2. The $(A, N)$ distance matrix is computed each iteration via broadcasting: `diff_an = attractors[:, None] - nodes_arr[None]`. NumPy's `np.argmin` finds the closest node per attractor in one call.
3. New nodes are grown for all active nodes in the same iteration (parallel growth), stored in temporary lists to avoid modifying `nodes` mid-loop.
4. All vein edges are drawn after the simulation via `LineCollection` with colors indexed by tree depth (distance from root) and linewidth proportional to $(1 - t)$ so the mid-rib is thickest.

**Interesting Properties**
The space colonization algorithm naturally produces the hierarchical venation architecture seen in real dicot leaves: a dominant mid-rib, secondary veins branching from it, and fine tertiary reticulation — all without any explicit programming of this hierarchy. Reducing the kill radius $r_k$ produces denser, more reticulate venation (similar to oak leaves); increasing it gives open, parallel venation (similar to grasses).

### How This Works — Fire Particle System

**Core Idea**
A fire particle system models combustion as a cloud of discrete embers, each born at the fuel source, rising through convection, cooling as they age, and fading into smoke. The visual impression of continuous flame emerges from rendering thousands of particles simultaneously at every life stage.

**Mathematics**
Each particle carries an age $a \in [0,1]$, drawn from a Beta$(1, 2.5)$ distribution so young particles (near $a=0$) are far more numerous than dying ones — matching the dense base of a real flame. Position is:
$$x = \mathcal{N}\!\left(0.5,\;\sigma_x(a)\right), \qquad y = a\,(h_{\max} + \delta), \quad h_{\max} = 0.70 + \text{heat}\times 0.22$$
where $\sigma_x(a) = 0.05 + \text{turbulence}\times 0.12\,a$ widens the flame horizontally with age. Colour follows a temperature-to-RGB mapping: green channel $g = \min(1, 1.9(1-a))$ and blue channel $b = \max(0,(1-a-0.65)\times 3)$, producing the white–yellow–orange–red gradient of a blackbody radiator.

**Logic in the Code**
1. `rng.beta(1.0, 2.5, n)` produces age values skewed toward 0, so most particles are near the base (hot and bright).
2. Horizontal spread is proportional to age — older particles have had more time to drift laterally via turbulence.
3. RGBA colour arrays are assembled with pure numpy broadcasting; no per-particle loop.
4. A separate ember cluster (tight Normal spread, age≈0) is overlaid at $y\approx 0$ to represent the hot fuel surface.
5. `ax.scatter` renders all particles in a single draw call with size inversely proportional to age.

**Interesting Properties**
The Beta distribution is the key to realism: a uniform age distribution would make the flame look hollow (equal density everywhere), while Beta$(1, 2.5)$ naturally concentrates particles at the base, matching the luminous cone shape visible in candle flames. Increasing turbulence shifts $\sigma_x$, transitioning from a calm pillar to a storm-tossed torch.

### How This Works — Galaxy Spiral Arms

**Core Idea**
Spiral galaxies exhibit arms that follow logarithmic spirals, a form that appears throughout nature wherever growth occurs at a constant angle relative to the centre. This pattern simulates a barred-spiral galaxy by placing star-like points along multiple logarithmic arms plus a compact central bulge.

**Mathematics**
A logarithmic spiral in polar form is:
$$r(\theta) = r_0\,e^{b\,\theta}$$
where $r_0 = 0.01$ is the inner radius and $b$ is derived so $r$ reaches $r_{\max}=0.46$ over $\theta_{\max} = \text{winding}\times 3\pi$:
$$b = \frac{\ln(r_{\max}/r_0)}{\theta_{\max}}$$
Each arm is rotated by $2\pi k/N_{\text{arms}}$ for arm index $k$. Stars are scattered with Gaussian spread $\sigma(t) = 0.004 + 0.020\,t$ perpendicular to the arm tangent $(-\sin\theta, \cos\theta)$, widening toward the outer edge.

**Logic in the Code**
1. For each arm, $t\in[0,1]$ is sampled uniformly; $\theta = t\,\theta_{\max}$ and $r = r_0 e^{b\theta}$.
2. Cartesian coordinates are computed then perturbed perpendicular to the local tangent direction.
3. A central bulge is added using a half-Normal radial distribution, producing a bright compact nucleus.
4. Star colour transitions from warm yellow-white ($t\approx 0$, inner) to cool blue-white ($t\approx 1$, outer), mimicking stellar population gradients.
5. Point sizes decrease with $t$ so inner stars appear brighter.

**Interesting Properties**
The winding number controls how many turns each arm makes. At high winding (many turns), arms overlap and the galaxy looks like a tight grand-design spiral (think M51); at low winding, it resembles a flocculent open structure. The logarithmic form means the pitch angle $\psi = \arctan(1/b)$ is constant along the arm — a property observed in real galaxies.

### How This Works — Aurora Borealis

**Core Idea**
The aurora borealis forms when charged solar-wind particles excite atmospheric gases along magnetic field lines, producing glowing curtains of light. This pattern simulates multiple translucent curtains using sinusoidally-shaped intensity masks rendered directly onto a raster image.

**Mathematics**
Each curtain's upper edge traces a compound wave:
$$y_{\text{edge}}(x) = y_0 + A\sin\!\left(\frac{2\pi f x}{W} + \phi\right) + 0.4A\sin\!\left(\frac{2\pi\cdot 2.3f x}{W} + 1.7\phi\right)$$
The pixel intensity at row $r$, column $c$ is:
$$I(r,c) = \exp\!\left(-\frac{\max(0,\,r - y_{\text{edge}}(c))}{L}\right) \cdot S(c)$$
where $L$ is the curtain length and $S(c)$ is a shimmer modulation factor. Curtains are composited additively onto the RGB canvas.

**Logic in the Code**
1. A difference matrix `dy[row, col] = row - wave[col]` is computed in one numpy broadcast, covering the full $H\times W$ grid.
2. The exponential decay is applied element-wise — no Python loop over rows.
3. A shimmer sinusoid multiplies each column, producing the flickering vertical striations of a real aurora.
4. Multiple curtains are composited additively, clipped, then gamma-corrected with a 0.55 power curve to lift dark regions.
5. A sparse starfield is painted last at full brightness, visible through the curtain gaps.

**Interesting Properties**
The additive compositing model means overlapping curtains of different colours mix to produce intermediate hues — green+purple yields cyan-tinted columns. Real auroras exhibit exactly this blending where oxygen (green) and nitrogen (blue/red) emission bands overlap.

### How This Works — Underwater Caustics

**Core Idea**
Caustics are the bright, branching light patterns visible on the floor of a swimming pool or shallow sea. They arise because a wavy water surface acts as a lens array, focusing refracted light into moving regions of high intensity. This pattern simulates the interference field that approximates the caustic distribution.

**Mathematics**
The intensity field is modelled as a sum of plane-wave cosines from $N$ random directions:
$$F(x,y) = \sum_{i=1}^{N} \cos(\mathbf{k}_i \cdot \mathbf{r} + \phi_i), \quad \mathbf{k}_i = 2\pi f\,(\cos\alpha_i,\, \sin\alpha_i)$$
where $\alpha_i \sim \mathcal{U}[0, 2\pi)$ and $\phi_i \sim \mathcal{U}[0, 2\pi)$. After normalising $F$ to $[0,1]$, a power law sharpens the peaks:
$$C(x,y) = F(x,y)^{\gamma}, \quad \gamma = \text{sharpness}$$
High $\gamma$ pushes the field toward binary (dark background, thin bright caustic threads).

**Logic in the Code**
1. The full $N\times N$ grid of $(x,y)$ coordinates is computed once via `np.meshgrid`.
2. Each wave's contribution is a single vectorised numpy expression — no spatial loops.
3. After summation the field is normalised, then `np.power` applies $\gamma$ uniformly.
4. The scalar field is mapped to a blue-teal RGB image by independently scaling each channel, giving the characteristic underwater blue cast with bright cyan highlights.

**Interesting Properties**
The interference pattern is deterministic given the wave directions but appears highly irregular — it is a quasi-random field whose statistics resemble real caustics (exponential intensity distribution). Increasing frequency raises the spatial density of caustic threads, while increasing sharpness $\gamma$ transitions the image from smooth gradients to an almost binary light-map.

### How This Works — Sand Dune Erosion

**Core Idea**
Desert sand dunes form and migrate through saltation: wind lifts grains from one location and deposits them downwind. Where the supply exceeds the angle of repose, avalanching redistributes excess sand to restore a stable slope. This pattern simulates both effects on a 2-D heightfield.

**Mathematics**
At each time step the saltation flux from cell $(i,j)$ to its downwind neighbour $(i,j+1)$ is:
$$q_{i,j} = w\,h_{i,j}$$
where $w$ is wind speed. Heights update as:
$$h \leftarrow h - q + \mathrm{roll}(q,\,+1,\,\mathrm{axis}=1)$$
The avalanche rule checks the right-neighbour slope $s_{i,j} = h_{i,j} - h_{i,j+1}$. If $s > \theta_r$ (angle of repose), half the excess transfers:
$$h_{i,j} \leftarrow h_{i,j} - \tfrac{1}{2}(s - \theta_r), \qquad h_{i,j+1} \leftarrow h_{i,j+1} + \tfrac{1}{2}(s - \theta_r)$$
Both operations use `np.roll` for periodic boundary conditions.

**Logic in the Code**
1. The heightfield is initialised with random noise plus a few pre-formed ridges to seed dune formation.
2. Each iteration applies the saltation step (one roll) followed by avalanche passes in both x and y directions.
3. Periodic boundaries (implicit in `np.roll`) let sand wrap around the canvas, simulating an infinite dune field.
4. The normalised heightfield is mapped to a sandy RGB palette: dark ochre in troughs, bright cream at crests — matching the shadowed face and sun-lit slip face of real barchans.

**Interesting Properties**
The interplay between saltation (directional transport) and avalanching (isotropic smoothing) spontaneously organises the random initial field into asymmetric dunes with a gentle windward slope and a steep leeward slip face — the characteristic barchan morphology. Increasing wind speed accelerates migration; reducing the avalanche threshold produces sharper, more angular crests.

### How This Works — Coral Reef Growth

**Core Idea**
Coral polyps grow by budding new individuals at the tips of existing branches, producing the characteristic fan, staghorn, and brain morphologies. This pattern captures that branching growth process using a depth-limited recursive tree whose parameters mimic specific coral species.

**Mathematics**
Each branch of length $\ell$ at depth $d$ spawns $n_{\text{sub}} \in \{2, 3\}$ children at angles evenly distributed over an angular fan $[-\phi, +\phi]$ around the parent direction $\alpha$:
$$\alpha_k = \alpha - \phi + \frac{2k\phi}{n_{\text{sub}}-1} + \varepsilon, \quad \varepsilon \sim \mathcal{U}[-0.08, 0.08]$$
Child length is $\ell_{\text{child}} = \ell \cdot U[0.57, 0.73]$. Recursion terminates at $d=0$ or $\ell < 0.004$. The fan-spread control $\phi \in [0.25,\,0.85]\cdot\pi/2$ governs whether growth is a tight pillar (small $\phi$) or a broad fan (large $\phi$).

**Logic in the Code**
1. An outer loop places $N_{\text{colonies}}$ colonies at random positions along the seafloor.
2. Each colony uses its own colour palette (6 species presets), passed via a default-argument capture to avoid closure aliasing.
3. Segments accumulate in shared lists across all colonies, then are drawn in one `LineCollection` call — far faster than individual `ax.plot` calls.
4. Line width is proportional to depth $d$, so trunks are thick and tips are hairline — matching the tapered morphology of real coral.

**Interesting Properties**
The fan-spread parameter shifts the topology from anisotropic (tree-like, upward) to nearly isotropic (brain coral-like, radial). At extreme fan spread approaching $\pi$, growth folds back on itself, producing the rounded lobes of brain corals like *Diploria* species.

### How This Works — Mushroom Spore Map

**Core Idea**
Mushroom mycelium spreads outward from spore germination points, and cross-sections of the colony boundary map closely to Voronoi cells. This pattern uses a Voronoi distance field to tile the canvas into organic cells, then adds concentric rings and layered-sine noise to create the fibrous, earthy texture of a spore microscopy image.

**Mathematics**
Given $N$ spore centres $\{\mathbf{p}_i\}$, the two nearest-neighbour distances at pixel $\mathbf{q}$ are $d_1(\mathbf{q})$ and $d_2(\mathbf{q})$ (via `scipy.spatial.cKDTree`). Three texture layers are combined:
$$\text{rings}(\mathbf{q}) = \tfrac{1}{2}\bigl(1 + \sin(60\pi f_r d_1)\bigr)$$
$$\text{edge}(\mathbf{q}) = \left(\frac{d_2 - d_1}{d_1 + 0.02}\right)^{0.35}$$
$$\text{field} = \bigl[\text{rings}\cdot(1-\beta) + \text{noise}\cdot\beta\bigr]\cdot\text{edge} + 0.18\,(i_{\text{cell}} \bmod 9)/9$$
where $\beta$ is the noise-blend slider and noise is a three-octave layered-sine approximation to Perlin noise.

**Logic in the Code**
1. The entire pixel grid is queried against the KD-tree in one batched call, returning the two nearest distances and indices.
2. The ring field, edge mask, and noise field are all computed as full $N\times N$ numpy arrays without any Python spatial loop.
3. The cell index $i_{\text{cell}} \bmod 9$ shifts each cell's brightness by a small constant, making individual spore territories visually distinct.
4. The final scalar field is mapped to a three-channel image with different linear ramps per channel, yielding the cream-to-ochre-to-brown mushroom palette.

**Interesting Properties**
When ring frequency is high and noise blend is zero, the pattern resembles tree growth rings. When noise blend is 1.0 the rings vanish and the image looks like a fungal mycelium network stained under a microscope. The two limits represent different biological interpretations of the same Voronoi topology.

### How This Works — Terrain Height Map

**Core Idea**
Terrain generation uses **fractal Brownian motion (fBm)** — a weighted sum of layered noise octaves at geometrically increasing frequencies and decreasing amplitudes. Each octave adds finer detail, producing the characteristic self-similarity of real terrain: mountain ranges resemble foothills, which resemble individual ridges. The finished heightfield is coloured using a **hypsometric** palette that maps elevation bands to geographically meaningful hues.

**Mathematics**
The heightfield is:
$$h(x,y) = \frac{1}{Z}\sum_{k=0}^{n-1} H^k \cdot \sin\!\bigl(2^k f_0\, x + \phi_{k,1}\bigr) \cdot \cos\!\bigl(2^k f_0\, y + \phi_{k,2}\bigr)$$
where $H \in (0,1)$ is the **roughness** (persistence) parameter, $f_0$ is the base frequency, $\phi_{k,1}$ and $\phi_{k,2}$ are per-octave random phases, and $Z = \sum_k H^k$ normalises the sum to $[0,1]$. When $H$ is small, higher octaves are heavily suppressed — producing smooth, low-frequency terrain. As $H \to 1$, all octaves contribute equally — producing rough, fractal surfaces. The Hurst exponent of this process is related to $H$ by $D = 3 - H$ (fractal dimension of the surface).

**Logic in the Code**
1. Each octave draws new random phase offsets from the seeded RNG, ensuring reproducible but visually distinct terrain for every seed value.
2. After the loop, the raw sum is normalised to $[0,1]$ and the elevation is mapped through four Boolean masks: water ($h \leq w$), lowland, midland, and highland — each with a distinct linear colour ramp per RGB channel.
3. `matplotlib.contour` draws the water shoreline at $h = w$ in blue and subtle elevation bands at five equally-spaced heights in translucent black.

**Interesting Properties**
The coastline shape produced by fBm is statistically similar to real coastlines, whose fractal dimension $D \approx 1.2$–$1.3$ was famously analysed by Mandelbrot. Decreasing the water-level slider submerges land until only mountain peaks protrude — the same mechanism that produces archipelagos and drowned river valleys on geologic timescales.

### How This Works — Waterfall Flow

**Core Idea**
A waterfall is water flowing off a cliff edge and falling under gravity to a pool below. The visual signature — a white veil of braided streams — arises from the competition between gravity (pulling straight down) and the Kelvin-Helmholtz instability of the water sheet (causing lateral oscillations). This pattern simulates the effect using individual stream paths driven by gravity with sinusoidal sway, overlaid on a rocky cliff background raster image.

**Mathematics**
Each stream is a discrete-time path. At step $i$, the particle at $(x_i, y_i)$ updates as:
$$x_{i+1} = x_i + A\sin(\omega i + \phi), \qquad y_{i+1} = y_i - s\bigl(1 + k(1 - y_i)\bigr)$$
where $A$ is the sway amplitude, $\phi$ is a per-stream random phase, $s$ is the step size, and the factor $(1 + k(1-y_i))$ provides gravitational acceleration — the stream speeds up as it falls. The sinusoidal term models lateral oscillation of the falling water sheet.

**Logic in the Code**
1. A rocky cliff background is built as an $N \times N$ raster image using five octaves of layered sine noise; the bottom 18% is recoloured dark blue to represent the pool.
2. Each of $n_{\text{streams}}$ particle paths is traced step by step, collecting segment endpoints into shared lists — never drawing mid-loop.
3. A single `LineCollection` renders all segments with per-segment RGBA colour (bright white-blue at the crest, slightly grey at the pool) and per-segment linewidth (thicker at the crest, hairline at the base).
4. A spray cloud uses `ax.scatter` with Beta$(1.5, 3.5)$-distributed $y$-coordinates, concentrating most spray near the pool surface with a few droplets drifting upward.

**Interesting Properties**
The acceleration factor $1 + k(1-y)$ means streams enter the pool faster than they leave the cliff edge, mimicking free-fall under gravity. Real waterfalls exhibit a **hydraulic jump** at the base: a rapid transition from fast, shallow flow to slow, deep flow. This is the physical origin of the turbulent spray cloud, and its radius scales with the incoming flow velocity squared.

### How This Works — Tornado Vortex

**Core Idea**
A tornado is a rapidly rotating column of air connecting a cumulonimbus cloud to the ground. The visible funnel is a condensation cloud that reveals the pressure drop inside the vortex core. This pattern simulates the funnel geometry by distributing particles in a tapered cylinder whose radius is narrow at the top and wide at the base, then twisting their angular positions proportionally to height.

**Mathematics**
In a Rankine vortex the tangential velocity is:
$$v_\theta(r) = \begin{cases} \Omega r & r \leq R_c \quad\text{(solid-rotation core)} \\ \Gamma / (2\pi r) & r > R_c \quad\text{(irrotational outer region)} \end{cases}$$
The funnel radius at normalised height $y \in [0,1]$ is:
$$R(y) = R_{\text{top}} + (1-y)(R_{\text{base}} - R_{\text{top}})$$
A particle at height $y$ receives a twist $\Delta\theta = \omega \cdot y \cdot 2\pi$, where $\omega$ is the Vortex Strength slider. Its 2D projected $x$-coordinate is $x = 0.5 + r\cos(\theta_0 + \Delta\theta)$.

**Logic in the Code**
1. Heights $y$ are drawn uniformly; the local funnel radius follows from the linear taper formula. Each particle's radial distance is a half-Normal centred on the funnel wall, placing debris at the condensation boundary.
2. The twist $\omega \cdot y \cdot 2\pi$ is added to the random azimuth before projection — higher $\omega$ wraps the spiral more tightly.
3. Colour encodes radius (bright white core to dark debris wall) modulated by height (darker near ground, lighter at cloud base), computed entirely with numpy broadcasting.
4. The funnel outline is two polylines $x = 0.5 \pm R(y)$, drawn faintly as a geometric guide.

**Interesting Properties**
Conservation of angular momentum explains why the funnel narrows upward: $r \cdot v_\theta = \Gamma/(2\pi) = \text{const}$, so $v_\theta$ must increase as $r$ decreases — the same mechanism that makes an ice skater spin faster when pulling in their arms. Setting the Funnel Top slider near zero approaches an idealised point vortex, while large values produce a nearly cylindrical pillar of cloud.

### How This Works — Cloud Formation

**Core Idea**
Cumulus clouds form when moist air rises, expands adiabatically, and cools to its dew point at which water vapour condenses into liquid droplets. The characteristic flat cloud base marks the condensation level; the rounded top is limited by convective instability. This pattern simulates the cloud water-content density field using multi-octave persistence noise and composites it over a physically motivated sky gradient.

**Mathematics**
The cloud density field is fractal Brownian motion:
$$N(x,y) = \frac{1}{Z}\sum_{k=0}^{n-1} H^k \cdot \sin\!\bigl(2^k f_0 x + \phi_{k,1}\bigr)\cdot\cos\!\bigl(2^k f_0 y + \phi_{k,2}\bigr)$$
Cloud opacity is derived by thresholding with a power-law lift:
$$\alpha(x,y) = \left(\frac{\max(0,\, N - \tau)}{1 - \tau}\right)^{1/2}, \qquad \tau = 1 - \text{density}$$
Larger density lowers $\tau$, exposing more of the noise field as cloud. The $\tfrac{1}{2}$ exponent softens the hard threshold edge, producing smooth billowing boundaries rather than binary cutoffs.

**Logic in the Code**
1. Each octave is a sine-cosine product with independent random phases; frequency doubles and amplitude scales by `persistence` each step.
2. The sky gradient is a 2D linear ramp — warm hazy horizon to deep blue zenith — computed as numpy arrays over the grid.
3. A shadow layer rolls the cloud alpha 10 rows and 7 columns to simulate sunlight from upper-left, darkening the sky beneath cloud edges by up to 22%.
4. Final compositing: $\text{pixel} = \text{sky} \cdot \text{shadow} \cdot (1-\alpha) + \text{cloud} \cdot \alpha$.

**Interesting Properties**
High persistence (near 0.85) produces large connected cloud masses resembling overcast stratus; low persistence (near 0.35) creates isolated wispy patches resembling cirrus. This maps to the physical correlation length of atmospheric turbulence: high persistence means large-scale convective cells dominate, while low persistence means small-scale granularity breaks up any continuous sheet.

### How This Works — River Delta Branching

**Core Idea**
River deltas form where a river meets standing water, decelerates, and deposits its sediment load. Flow splits into distributary channels wherever a channel becomes too shallow to carry its full discharge — a process called **avulsion**. This pattern models the branching network as a recursive binary tree growing downward from a single trunk, with a fan-spread parameter controlling delta morphology from bird's-foot to arcuate.

**Mathematics**
Each channel of length $\ell$ at recursion depth $d$ bifurcates into two children at angles $\pm\phi/2$ from the parent heading $\alpha$:
$$\alpha_{\text{L}} = \alpha - \phi/2 + \varepsilon, \quad \alpha_{\text{R}} = \alpha + \phi/2 + \varepsilon, \quad \varepsilon \sim \mathcal{U}(-0.06, 0.06)$$
Child length is $\ell' = \ell \cdot r + \eta$ where $r$ is the Length Decay slider. The downward convention is $\Delta x = \ell\sin\alpha$, $\Delta y = -\ell\cos\alpha$, so $\alpha = 0$ is straight down. With depth $d$, the maximum terminal channels is $2^d$.

**Logic in the Code**
1. The recursive `_channel` function accumulates segment data (endpoints, depth fraction $t$, linewidth) into shared lists — no drawing during recursion, keeping geometry separate from rendering.
2. After recursion, a single `LineCollection` draws all channels with per-segment colour: muddy-brown at $t=0$ (trunk) to teal-blue at $t=1$ (sea-reaching distributaries).
3. A sediment fan scatter uses Beta$(1.2, 2.8)$-distributed $y$-coordinates to concentrate deposits near the shoreline.
4. A 28% probability tertiary branch introduces realistic mid-channel avulsions, breaking perfect binary symmetry for a more natural appearance.

**Interesting Properties**
The Fan Spread parameter governs delta morphology: small values produce bird's-foot deltas (like the Mississippi — high sediment supply, low wave energy); large values produce arcuate fan deltas (like the Nile — wave energy redistributes sediment laterally). At the limit $\phi \to \pi$, the tree folds back on itself and loses all net downstream transport.

### How This Works — Moth Wing Pattern

**Core Idea**
Moth wing patterns serve dual ecological functions: **cryptic coloration** (earthy brown-ochre palettes mimic tree bark and dead leaves) and **deimatic display** (prominent eyespots startle predators or direct their strike away from the body). The coloration is produced during metamorphosis by spatially regulated pigment gene expression, generating both concentric band structures and sharply defined eyespot rings.

**Mathematics**
The wing shape is the union of two ellipses. The concentric band field uses the minimum elliptical distance $D = \min(d_{\text{fore}},\, 1.05 \cdot d_{\text{hind}})$:
$$B(x,y) = \tfrac{1}{2}\bigl(1 + \sin(D \cdot k \cdot \pi \cdot s)\bigr)$$
where $k$ is band count and $s$ is pattern scale. Each bilateral eyespot uses Gaussian radial ring functions:
$$E_j(r) = \exp\!\left(-\frac{(r - r_j)^2}{2\sigma_j^2}\right), \quad r = \|(x,y) - (e_x,e_y)\|$$
with four concentric rings: outer light ring, dark ring, inner iris, and dark pupil.

**Logic in the Code**
1. The base field is `bands * 0.65 + noise * 0.35`, mapped to a tan-to-brown RGB palette via three independent linear ramps per channel.
2. Wing veins subtract a narrow Gaussian strip along each radial direction from the body centre: `exp(-perp_dist^2 * 1800) * forward`, multiplied by the wing mask to prevent bleed-through.
3. Eyespots add and subtract four Gaussian ring contributions per spot using `np.where` with the combined wing and radius mask.
4. A bilateral eyespot at $(+0.45, e_y)$ and $(-0.45, e_y)$ for each row in `eye_ys` enforces left-right symmetry — exactly the developmental mechanism used in real Lepidoptera.

**Interesting Properties**
Eyespots function as **deflection marks**: experimental studies with predatory birds show they aim at the prominent false eyes rather than the moth's body, causing survivable wing damage instead of lethal strikes. The concentric ring structure arises from a reaction-diffusion morphogen gradient during wing disc development — the same Turing-pattern mechanism as Pattern 23, operating at millimetre scale within each scale cell.

---

## Section 5 — Export Utilities

Export functions are available via the **💾 EXPORT** button above.
- `export_png(figure, name)` — saves as PNG
- `export_gif(frames, name, fps)` — saves as animated GIF
- `export_mp4(frames, name, fps)` — saves as MP4 (falls back to GIF if ffmpeg unavailable)

All outputs are saved to `visual_engine/exports/`.

### How This Works — Generative Mondrian

**Core Idea**
Piet Mondrian's iconic grid paintings reduce composition to its essentials: thick black lines dividing a white canvas into rectangles, with a few cells flooded in primary red, blue, or yellow. Generative Mondrian recreates this by treating the canvas as a recursive binary space partition tree.

**Mathematics**
At each step every rectangle $(x, y, w, h)$ is split with probability $p = 0.75$. The cut is placed at a random fraction $f \sim \text{Uniform}(0.25,\, 0.75)$ along the longer axis:

$$x_{\text{cut}} = x + f \cdot w \quad \text{(horizontal split if } w \ge h\text{)}$$

After $s$ split rounds the expected number of rectangles grows as $O\bigl((1.5)^s\bigr)$ — exponential growth pruned by the 25 % no-split probability.

**Logic in the Code**
1. A list of rectangles starts with a single unit square.
2. Each iteration replaces every rectangle stochastically: if split, two child rectangles are appended; otherwise the parent is kept unchanged.
3. After all splits, each rectangle is drawn as a `matplotlib.patches.Rectangle` with black edge and `line_width` controlling stroke weight.
4. Colour is sampled from the Mondrian palette (5 whites, 1 red, 1 blue, 1 yellow), so the 5:3 weighting keeps most cells white, matching Mondrian's own compositional ratios.

**Interesting Properties**
Because each split is independent, the composition is statistically self-similar at every scale — zooming into any corner tends to show the same mix of large white and small coloured rectangles. Changing `seed` produces an entirely different layout while preserving this statistical character.


### How This Works — Perlin Noise Painting

**Core Idea**
Fractal Brownian motion (fBm) noise produces smooth, naturalistic textures — clouds, terrain, marble, wood grain — by layering octaves of coherent noise at increasing frequencies and decreasing amplitudes. Unlike white noise, nearby pixels are correlated, giving the characteristic organic look.

**Mathematics**
The fBm field sums $n$ octaves:

$$F(x, y) = \sum_{k=0}^{n-1} \frac{1}{2^k} \cdot N_k\!\left(2^k x,\; 2^k y\right)$$

where $N_k$ is smooth (value) noise at octave $k$. Persistence $= 0.5$ halves amplitude each octave; lacunarity $= 2$ doubles spatial frequency. The series converges because $\sum_{k=0}^{\infty} (1/2)^k = 2$.

**Logic in the Code**
1. For octave $k$, a small random grid of size $(\texttt{scale} \cdot 2^k + 2)^2$ is drawn from `rng.random()`.
2. `scipy.ndimage.zoom` upsamples this grid to full resolution using cubic interpolation (`order=3`), creating spatially correlated smooth values.
3. Each octave is accumulated with weight $1/2^k$; the final result is normalised to $[0, 1]$.
4. The scalar field is rendered via a `LinearSegmentedColormap` built from the chosen palette, mapping noise value to colour.

**Interesting Properties**
The `scale` control changes the base spatial frequency — high values zoom in to show large smooth blobs; low values produce fine-grained detail. Beyond 7–8 octaves, visual improvement is negligible because the highest-frequency layers contribute less than 1 % of total signal energy.


### How This Works — Mandala Generator

**Core Idea**
Mandalas are circular diagrams with $n$-fold rotational symmetry used in Hindu and Buddhist traditions as meditation aids. Computationally they are built by repeating a single petal motif $n$ times around a central axis, then nesting multiple such rings at increasing radii.

**Mathematics**
Each petal in ring $r$ occupies an angular sector of half-width $\delta = \pi/n$. Its radial profile follows a cosine-squared envelope:

$$\rho(\theta) = r_{\text{base}} + r_{\text{tip}} \cdot \cos^2\!\!\left(\frac{(\theta - \theta_k) \pi}{2\delta}\right)$$

where $\theta_k = 2\pi k/n$ is the centre angle of petal $k$. This produces a smooth lens-shaped outline that peaks at $\theta_k$ and tapers to zero at the sector boundaries.

**Logic in the Code**
1. For each ring, `r_base` grows linearly with ring index; odd rings are phase-shifted by $\pi/n$ so their petals interleave with even rings.
2. For each of the $n$ copies, $\rho(\theta)$ is evaluated over 80 angular points and converted to Cartesian $(x, y) = \rho \cdot (\cos\theta,\sin\theta)$.
3. `ax.fill` renders the petal interior; `ax.plot` adds an outline in a contrasting palette colour.
4. Dot accents are placed at the tip and root radii to add cross-ring connective detail.

**Interesting Properties**
Prime symmetry values (5, 7, 11) prevent petals from overlapping between rings, giving an open, airy design. Even values (6, 8, 12) produce denser, jewel-like compositions because alternate rings phase-lock their petals.


### How This Works — Stained Glass Voronoi

**Core Idea**
A Voronoi diagram partitions the plane into cells, each containing all points closer to one seed than to any other. When coloured and given thick black borders the cells resemble the leaded panes of a stained glass window — each "pane" a different transparent colour, the borders the metal came that holds them.

**Mathematics**
For seed points $\{p_i\}$, the Voronoi cell of seed $p_i$ is:

$$V_i = \bigl\{\, x \in \mathbb{R}^2 \;\big|\; \|x - p_i\| \le \|x - p_j\|\;\; \forall j \ne i \,\bigr\}$$

A border pixel satisfies $d_2(x) - d_1(x) < \varepsilon$, where $d_1, d_2$ are the distances to the nearest and second-nearest seeds and $\varepsilon = \texttt{border\_width} / \text{res}$ scales with render resolution.

**Logic in the Code**
1. `scipy.spatial.cKDTree.query(grid, k=2)` finds the two nearest seeds for every pixel in a single vectorised call.
2. Each pixel receives the palette colour of its nearest seed.
3. A shading term $1 - 0.28 \cdot d_1 / d_{\max}$ darkens pixels far from their seed, producing a convex-lens highlight that mimics light refraction through glass.
4. Pixels satisfying the border condition are set to black, drawing the leading lines.

**Interesting Properties**
`border_width` directly controls perceived lead thickness. Below 1 the borders disappear and the image becomes a flat colour mosaic; above 6 the glass panes are nearly invisible and the image reads as a pure line network.


### How This Works — Op-Art Optical Illusion

**Core Idea**
Op-Art (Optical Art) exploits the visual system's edge-detection and motion-processing pathways to create illusions of vibration, depth, and movement from purely static images. The key is high-contrast alternating patterns whose edges are too closely spaced for the eye to track without apparent motion.

**Mathematics**
**Style 0 (wavy rings):** Ring $i$ has radius perturbed by a sinusoid:

$$r(\theta) = r_i + A \sin(f\,\theta), \qquad r_i = 0.05 + \frac{i}{n}$$

where $A$ is `wave_amp` and $f$ is `wave_freq`. Painter's-algorithm fill from outside to inside creates alternating black/white bands.

**Style 2 (concentric squares):** Each side is displaced perpendicular to its edge by $d(t) = A\sin(f\pi t)$, where $t \in [-1,1]$ runs along the half-side length $s$.

**Logic in the Code**
1. Rings are drawn from outermost to innermost with `ax.fill`, exploiting the painter's algorithm so each ring occludes the previous.
2. `i % 2` alternates the fill colour, producing the high-contrast stripe sequence that drives the optical effect.
3. Style 1 draws a 9×9 grid of concentric ring pairs; the inner dark disc against the outer bright ring creates a Vasarely "bull's eye" that distorts into apparent depth across the grid.

**Interesting Properties**
The vibration illusion is strongest when stripe width matches the peak spatial frequency of foveal ganglion cells — roughly 20–30 rings for a 7-inch figure at arm's length. Too few rings produce simple stripes; too many exceed visual acuity and the image appears uniformly grey.


### How This Works — Watercolor Wash Effect

**Core Idea**
Traditional watercolour painting lays translucent pigment washes that pool and dry with soft, irregular edges. The procedural approximation simulates this by compositing multiple semi-transparent colour layers, each shaped by a randomly placed ellipse mask softened with Gaussian blur.

**Mathematics**
Layer $\ell$ accumulates $m$ random ellipses into a coverage mask. An ellipse at $(c_x, c_y)$ with semi-axes $(r_x, r_y)$ and rotation $\alpha$ contributes the indicator:

$$M_\ell \mathrel{+}= \mathbf{1}\!\left[\left(\frac{x'}{r_x}\right)^{\!2} + \left(\frac{y'}{r_y}\right)^{\!2} < 1\right]$$

where $x' = (x-c_x)\cos\alpha + (y-c_y)\sin\alpha$. After Gaussian smoothing with radius $\sigma$, the mask is raised to the $0.65$ power, stretching midtones and sharpening the soft-edged bead typical of dried watercolour pigment.

**Logic in the Code**
1. For each layer, $m$ random ellipses are rasterised onto a float accumulator using numpy boolean indexing.
2. `scipy.ndimage.gaussian_filter` blurs the accumulator, producing continuous opacity gradients.
3. The $0.65$ power-law curve is applied; the layer is then alpha-composited ($C' = C(1-\alpha) + C_\ell\alpha$, $\alpha = 0.55 M$) onto the RGBA canvas.
4. A final multiplicative noise layer in $[0.94, 1.0]$ simulates cold-press paper grain.

**Interesting Properties**
The `blur_sigma` parameter is the dominant control: small values produce hard ink-like edges; large values create diffuse washes. The $0.65$ exponent controls edge crispness: values below $0.5$ yield sharp acrylic-like blobs; values above $1.0$ produce uniformly soft oil-glaze transitions.


### How This Works — Glitch Art Generator

**Core Idea**
Glitch art appropriates digital corruption artefacts — the visual noise of mis-decoded image data — as an aesthetic medium. This generator simulates three canonical digital failure types: horizontal row displacement, chromatic aberration, and corrupted data blocks.

**Mathematics**
**Row displacement:** Slice $[r_0, r_0+h)$ is cyclically shifted right by $s$ columns:

$$\text{out}[i,\; j] = \text{base}\!\left[i,\; (j - s) \bmod W\right]$$

This mimics a video frame-sync failure where the scan beam jumps to the wrong horizontal position mid-frame.

**Chromatic aberration:** RGB channels are offset independently by $\pm\delta$ columns:

$$R' = \operatorname{roll}(R, +\delta), \quad G' = G, \quad B' = \operatorname{roll}(B, -\delta)$$

This replicates lateral chromatic aberration of uncorrected lenses or JPEG sub-pixel mis-registration.

**Logic in the Code**
1. A base image is built from horizontal palette fades blended with a vertical sine interference pattern, giving the glitch something textured to distort.
2. `n_slices` random horizontal bands are shifted by random integers via `np.roll` — cheap and vectorised.
3. After row-shifting, RGB channels are rolled in opposite directions to add chromatic fringing.
4. Random rectangular blocks are overwritten with single palette colours, simulating corrupted MPEG macroblock data.
5. Even rows are multiplied by $0.88$ to simulate CRT scanline interlacing.

**Interesting Properties**
`shift_max` is the dominant aesthetic control: values below 20 produce subtle wobble; above 150 the image fragments into barely recognisable bands. Setting `channel_shift` to 0 isolates the row-shift effect and reveals the underlying block structure most clearly.


### How This Works — Isometric City Builder

**Core Idea**
An isometric city builder renders a grid of rectangular buildings using a parallel-projection
view where the three coordinate axes appear equally foreshortened. Because there is no
perspective vanishing point, all lines along the same axis remain parallel, giving the
composition a clean, game-board aesthetic reminiscent of classic strategy titles like
*SimCity 2000*.

**Mathematics**
The isometric transform maps 3-D grid coordinates $(c, r, z)$ to 2-D screen
coordinates $(s_x, s_y)$:

$$s_x = (c - r) \cdot w, \qquad s_y = (c + r) \cdot h + z \cdot d$$

where $w = 1$ is the half-tile width, $h = 0.5$ is the half-tile height ratio, and
$d = 0.7$ is the vertical scale per floor. Three faces are visible: the **top** face
(constant $z = H$), the **right** face (constant $c = c_0 + 1$), and the **left**
face (constant $r = r_0 + 1$). Each face is a parallelogram in screen space.

**Logic in the Code**
1. For each grid tile a random integer height $H \in [1, H_{\max}]$ and a palette
   colour index $t \in [0,1]$ are sampled.
2. The draw order is determined by sorting tiles in decreasing $(c + r)$ sum —
   the **painter's algorithm** — so that buildings furthest from the viewer are
   painted first and naturally occluded by nearer buildings.
3. For each building the three face polygons are computed by calling `to_iso` on the
   eight relevant corners, then added as `patches.Polygon` with brightness multipliers
   1.35 / 0.72 / 0.46 for top / right / left faces to simulate a single overhead
   light source.
4. Window dots are placed at every other floor on the right face as yellow scatter markers.

**Interesting Properties**
Changing `seed` while holding `grid_size` and `max_height` fixed reshuffles only heights
and colours, preserving the overall silhouette shape. The painter's algorithm only works
correctly when buildings do not overlap in screen space; for a uniform grid of rectilinear
buildings this is always guaranteed by the monotone $(c + r)$ ordering.

### How This Works — Circuit Board Art

**Core Idea**
A printed circuit board (PCB) is a laminated sandwich of conductive copper traces on an
insulating substrate. The visual language — right-angle traces, circular annular rings,
component outlines in silkscreen white — is instantly recognisable and makes for compelling
generative art when the topological rules are respected.

**Mathematics**
The layout operates on a **snap grid** with pitch $\Delta = 0.04$ (normalised units),
so every trace endpoint satisfies $x = k\Delta$ for integer $k$. An L-shaped trace
connecting $(x_0, y_0)$ to $(x_1, y_1)$ is the union of two axis-aligned segments:
$\{(x,\, y_0) : x \in [x_0, x_1]\}$ and $\{(x_1,\, y) : y \in [y_0, y_1]\}$.
A via (through-hole) is represented as two concentric circles: outer radius $r_{\text{pad}}$
in copper gold and inner radius $r_{\text{hole}} = 0.42\,r_{\text{pad}}$ as the drill hole.

**Logic in the Code**
1. Each trace is generated as either a straight horizontal/vertical line or an L-shaped
   polyline after snapping both endpoints to the grid.
2. Vias are drawn as `patches.Circle` pairs — the outer pad in copper gold, the drill hole
   in board green.
3. IC chips are `FancyBboxPatch` rectangles with pin stubs extending left and right, plus a
   silkscreen dot for pin-1 identification.
4. Four corner mounting holes complete the mechanical feel of a real PCB outline.

**Interesting Properties**
Real PCBs use design-rule checks (DRC) to prevent trace overlaps and maintain clearance.
This renderer deliberately ignores DRC, allowing traces to cross and overlap — which
actually enhances the visual density. Increasing `n_traces` reveals how random L-routing
quickly fills the board with a convincing copper-pour texture.

### How This Works — Tie-Dye Diffusion

**Core Idea**
Tie-dye patterns emerge from folding fabric at multiple points before immersing it in dye:
the folds act as radial centres around which colour diffuses outward in concentric rings.
Unfolding reveals an interference pattern of overlapping ripple systems — exactly what
this renderer models analytically.

**Mathematics**
For $N$ centres at positions $(c_i, r_i)$ with weights $w_i$, the field value at
pixel $(x, y)$ is:

$$f(x,y) = \sum_{i=1}^{N} w_i \cos\!\bigl(2\pi\, f_0\, d_i + \tau\, \theta_i \bigr)$$

where $d_i = \sqrt{(x-c_i)^2+(y-r_i)^2}$ is the Euclidean distance to centre $i$,
$\theta_i = \arctan2(y-r_i,\, x-c_i)$ is the polar angle, $f_0$ is the radial frequency
(rings per unit distance), and $\tau$ is a **twist** parameter that adds spiral chirality
by mixing the angular coordinate into the phase.

**Logic in the Code**
1. A normalised coordinate grid $[0,1]^2$ is meshed at the target resolution.
2. For each centre, $d_i$ and $\theta_i$ are computed via numpy broadcasting in a single
   vectorised pass.
3. The cosine superposition is accumulated into a single float array; the final field is
   rescaled to $[0,1]$ and passed to a matplotlib colormap via `imshow`.

**Interesting Properties**
Setting $\tau = 0$ gives clean concentric circles. Increasing $\tau$ introduces
Archimedean spiral arms. At $\tau \approx \pi$ the spirals complete a full half-turn
per radial cycle, producing a sunflower-like phase map. The interference minima and maxima
trace iso-phase curves of the combined wave field — a 2-D analogue of acoustic beats.

### How This Works — Geometric Collage

**Core Idea**
Cut-paper collage — layering torn and cut shapes of coloured paper to build a composition
— has been a major strand of modern art from Cubist papiers collés through Matisse's
*Jazz* cut-outs to contemporary graphic design. This renderer generates a procedural
analogue: each "paper shape" is a randomly positioned, randomly rotated primitive sampled
from a palette, composited with partial transparency to create luminous overlapping regions.

**Mathematics**
Each shape is a closed polygon defined by vertices

$$\mathbf{v}_k = \begin{pmatrix} c_x \\ c_y \end{pmatrix}
+ \mathbf{R}(\phi)\,\mathbf{p}_k, \qquad
\mathbf{R}(\phi) = \begin{pmatrix}\cos\phi & -\sin\phi \\ \sin\phi & \cos\phi\end{pmatrix}$$

where $\mathbf{p}_k$ are the canonical vertices and $\phi$ is a uniform random rotation.
Alpha compositing uses the Porter-Duff **over** operator:
$C_{\text{out}} = \alpha C_{\text{fg}} + (1-\alpha)C_{\text{bg}}$.

**Logic in the Code**
1. $N$ shapes are drawn in random order; later shapes occlude earlier ones.
2. Rectangles are explicitly rotated by applying the $2\times 2$ rotation matrix to their
   four corners — this avoids matplotlib's anchor-corner rotation ambiguity.
3. The 5-pointed star alternates outer radius $r$ and inner radius $0.4r$ across 10
   vertices computed at equally-spaced angles.
4. Partial transparency (`alpha` control) turns solid shapes into translucent layers,
   revealing the composition beneath.

**Interesting Properties**
Reducing `alpha` toward 0.2 shifts the aesthetic from bold cut-paper toward a watercolour
wash; raising it to 1.0 gives hard-edged Matisse-style shapes. Because shapes are drawn
in random sequence, the visual rhythm changes entirely with `seed` even at fixed
`n_shapes`.

### How This Works — Pixel Sorting Art

**Core Idea**
Pixel sorting is a glitch-art technique popularised by artist Kim Asendorf: pixels in an
image are rearranged within each row (or column) according to their brightness, hue, or
saturation, but only within contiguous spans that exceed a threshold. The result stretches
high-contrast regions into luminous smears while leaving low-contrast areas intact —
creating a characteristic interplay of order and chaos.

**Mathematics**
Let $L_j$ be a sort key (e.g., luminance $L = 0.299R + 0.587G + 0.114B$) for pixel $j$
in a row. A **span** is a maximal contiguous run with $L_j > \tau$. Within each span
$[a, b]$ pixels are reindexed by the permutation $\sigma$ that sorts their keys:

$$P'_j = P_{\sigma(j)}, \quad j \in [a, b], \quad
\sigma = \operatorname{argsort}\bigl(L_{a:b}\bigr).$$

Pixels outside any span are left unchanged. The threshold $\tau$ controls how much of
each row participates in sorting.

**Logic in the Code**
1. A multi-octave value-noise field is generated and mapped to RGB via a single vectorised
   `cmap(field)` call.
2. For each row a boolean mask `above = keys > tau` is walked with a state machine to find
   span boundaries.
3. Within each span, `np.argsort` produces the reordering permutation and fancy indexing
   applies it in one numpy operation.
4. Setting `vertical=1` rotates the image 90° before sorting and back afterward, sorting
   along columns instead of rows.

**Interesting Properties**
The threshold is the primary artistic lever: near 0 almost every pixel is sorted,
collapsing each row into a smooth gradient; near 1 only bright highlights are sorted,
producing subtle vertical streaks. The `sort_mode` parameter (luminance / red /
saturation-proxy) produces markedly different streak textures from the same underlying
image.

### How This Works — ASCII Art Renderer

**Core Idea**
ASCII art exploits the varying **optical density** of printable characters to approximate
a greyscale image using only text. Dense characters like `@`, `#`, `%` read as dark;
sparse ones like `.`, ` ` read as light. The renderer builds a two-octave noise field at
character resolution and maps each cell's brightness to the nearest character in a
10-level density ramp.

**Mathematics**
The luminance-to-character mapping is a staircase quantiser:

$$\text{char}(v) = \texttt{RAMP}\!\left[\min\!\left(\lfloor v \cdot L \rfloor,\; L-1\right)\right]$$

where $v \in [0,1]$ is the normalised brightness and $L = |\texttt{RAMP}|$ is the ramp
length. The noise field is a weighted sum of two value-noise octaves:

$$f = 0.65\,\mathcal{N}_{\text{coarse}} + 0.35\,\mathcal{N}_{\text{fine}}$$

where each $\mathcal{N}$ is a bicubically upsampled random grid.

**Logic in the Code**
1. Two random grids (coarse and fine) are upsampled to `(nrows, ncols)` via
   `scipy.ndimage.zoom` with cubic interpolation.
2. A vectorised `cmap(field)` call produces the RGBA colour array in one pass; the loop
   reads pre-computed colours rather than calling the colormap per character.
3. Each character is drawn with `ax.text` in monospace font; `nrows = ncols // 2`
   corrects for the 2:1 character aspect ratio so the output fills a square figure.

**Interesting Properties**
The `cols` slider adjusts rendering resolution: at low values (20–30) individual character
blocks dominate; at high values (80–90) density variations blend into a smooth tonal
image. Enabling `invert` swaps dark and light, which effectively exchanges the roles of
spaces and dense characters — dense regions become bright halftone highlights.

### How This Works — Kandinsky Color Study

**Core Idea**
Wassily Kandinsky believed colour and geometric form could evoke emotion independent of
representational content — the founding thesis of abstract art. His compositions combine
circles (spiritual completeness), triangles (dynamic tension), lines (force and direction),
and arcs (motion and yearning) in densely layered arrangements. This renderer procedurally
generates compositions in three Kandinsky stylistic modes.

**Mathematics**
For a fan of $n$ radiating spokes from $(c_x, c_y)$, spoke $k$ runs at angle

$$\alpha_k = \alpha_0 + \frac{k\,\pi}{n}, \quad k = 0, \ldots, n-1,$$

giving a half-turn spread of evenly-spaced directions. The rotated rectangle is produced
by applying the standard rotation matrix

$$\mathbf{R}(\phi) = \begin{pmatrix}\cos\phi & -\sin\phi \\ \sin\phi & \cos\phi\end{pmatrix}$$

to the axis-aligned half-extents $(\pm w/2,\, \pm h/2)$ then translating to $(c_x, c_y)$.

**Logic in the Code**
1. A `style` selector chooses background colour and palette: `style=0` targets
   *Composition VIII* (dark ground, hard primaries); `style=1` approximates
   *Yellow-Red-Blue* (warm ground, saturated primaries); `style=2` uses the
   user-selected colormap.
2. Seven element types are sampled uniformly: filled circle, concentric halo, triangle,
   bold stroke, rotated rectangle, arc, and spoke fan.
3. All rotations use explicit numpy matrix multiplication, guaranteeing centre-pivoted
   transforms for every shape type.
4. Elements are drawn in random order so later ones occlude earlier ones, creating
   layering analogous to impasto oil-paint application.

**Interesting Properties**
Keeping `n_elements` below 30 with `style=1` tends to produce spare compositions
resembling Kandinsky's early improvisations; above 80 the canvas saturates into a dense
all-over pattern closer to his late Bauhaus period. The concentric halo element directly
echoes his essay *Concerning the Spiritual in Art*, where the circle is described as the
most primordial geometric element.

### How This Works — Zentangle Automaton

**Core Idea**
Zentangle is a meditative structured art form built from repetitive pen-drawn patterns called *tangles*. The Automaton aspect means each cell of the canvas independently selects its tangle from a finite rule set — analogous to a cellular automaton where local, deterministic rules produce global visual complexity.

**Mathematics**
The canvas is partitioned into a $g \times g$ grid of unit cells. Each cell independently draws one of $k$ tangle patterns (controlled by `n_patterns`) via a discrete uniform random draw seeded by `seed`. The spiral tangle follows an Archimedean spiral:
$$r = r_0 + a\theta, \quad \theta \in [0,\, 4\pi], \quad a = \frac{0.42}{4\pi}$$
producing approximately two full turns. Diagonal lines are parallel strokes at offsets $\delta_k = k \cdot 0.20$ for $k \in \{-4,\ldots,4\}$. Concentric circles are placed at $r_j = 0.09 + 0.35(j-1)/4$ for $j = 1, \ldots, 5$.

**Logic in the Code**
For each cell $(c, r)$ a background rectangle is added using a palette colour sampled at $t \sim \mathcal{U}(0.1, 0.9)$. Then, depending on the randomly chosen tangle index, one of six drawing routines places pattern elements in data coordinates aligned to the cell boundary: (0) horizontal lines, (1) concentric circles, (2) dot grid, (3) cross-hatch, (4) diagonal lines, (5) Archimedean spiral. Because all coordinates are in the same axis data-space as the grid, shapes fall naturally within their cells with no clipping required.

**Interesting Properties**
Setting `n_patterns=2` and `grid_size=12` creates a binary rule layout that closely resembles a stochastic cellular automaton state map. Increasing `grid_size` shrinks the visible detail of each tangle but reveals emergent large-scale texture from the contrast between adjacent cells. Palettes with high contrast (e.g. *Monochrome*) make the tangle structure more legible; saturated palettes shift attention to colour rather than line.


### How This Works — Neon Sign Generator

**Core Idea**
Neon signs produce visible light by passing electrical current through sealed glass tubes filled with noble gases (neon emits red-orange; argon with a mercury coating emits blue-violet). The visual signature is a saturated, glowing core surrounded by a soft atmospheric bloom caused by scattered photons. This renderer simulates that bloom using additive multi-pass overdrawing.

**Mathematics**
The glow profile approximates a Gaussian point-spread function. For $n_g$ glow layers, layer $k$ (from $k = n_g$ down to $k = 1$) is drawn with:
$$w_k = w_{\text{base}}\bigl(1 + 3.5\,k/n_g\bigr), \quad \alpha_k = 0.055 + 0.18\,(1 - k/n_g)$$
Outermost layers are wide and nearly transparent; inner layers are narrower and more opaque. A final white pass at $0.38\,w_{\text{base}}$ adds the bright tube core.

**Logic in the Code**
The `neon_draw` helper accepts any $(x, y)$ array and applies the multi-pass glow loop before drawing the white core. Six shape generators feed it: circle, rectangle, zigzag polyline, sinusoidal wave $y = A\sin(f\pi(x-c_x)/s)$, star polygon with alternating outer/inner radii $R$ and $0.42R$, and arc. Each sign is assigned a random `base_lw` from $[2.0, 4.5]$ to simulate different tube diameters.

**Interesting Properties**
Setting `glow_layers=2` sharpens the signs toward plain coloured lines, removing the atmospheric bloom. Overlapping signs exhibit additive brightening at their intersection — exactly the effect visible when two real neon tubes cross. The fixed neon colour palette (“#ff073a”, “#04d9ff”, “#39ff14”, “#bc13fe”, etc.) bypasses the dropdown palette to preserve authentic noble-gas emission colours.


### How This Works — Mosaic Tile Art

**Core Idea**
Byzantine and Roman mosaic art builds images from small individually-coloured stone or glass tesserae separated by mortar (grout). The eye integrates the discrete patches at viewing distance through simultaneous contrast and spatial averaging — the same mechanism exploited by pointillism. This renderer produces a procedural mosaic where tile colours are driven by a continuous noise field.

**Mathematics**
A canvas of $W \times W$ pixels is partitioned into tiles of size $t_s$ pixels:
$$n_c = \left\lfloor \frac{W}{t_s} \right\rfloor, \quad n_r = \left\lfloor \frac{W}{t_s} \right\rfloor$$
Each tile $(r_i, c_j)$ occupies pixels $[c_j t_s + g,\; c_j t_s + t_s - g) \times [r_i t_s + g,\; r_i t_s + t_s - g)$ where $g$ is the grout width. Tile colour is:
$$\text{color}_{ij} = \text{cmap}\bigl(\text{clip}(F_{ij} + \varepsilon_{ij},\, 0, 1)\bigr), \quad \varepsilon_{ij} \sim \mathcal{U}(-0.06, 0.06)$$
Three colour modes determine $F$: (0) cubic-upsampled value noise, (1) radial distance gradient, (2) pure random.

**Logic in the Code**
The raster image array is initialised to the grout colour. For each tile the pixel block is filled with `tile_rgb = clip(cmap_rgb × b, 0, 1)` where $b \sim \mathcal{U}(0.90, 1.06)$ adds per-tile brightness jitter simulating the irregular reflectivity of hand-cut stone. The final `imshow` renders the full image in one call.

**Interesting Properties**
Setting `grout_width=0` removes grout lines, producing a continuous pixelated image reminiscent of low-resolution pixel art. Large `tile_size` values reduce the tessera count and reveal more noise structure. The radial mode (`color_mode=1`) produces concentric-ring layouts typical of Roman floor medallions.


### How This Works — Impressionist Dots

**Core Idea**
Pointillism (developed by Georges Seurat and Paul Signac, c. 1886) applies paint in discrete dots rather than brushstrokes, exploiting *optical colour mixing*: at sufficient viewing distance the eye integrates neighbouring dots into a perceived intermediate colour. Seurat’s theory drew on Ogden Rood’s chromatic circle and Chevreul’s simultaneous contrast laws.

**Mathematics**
A multi-octave fBm colour field $F: [0,1]^2 \to [0,1]$ is sampled on a $200 \times 200$ grid:
$$F = \frac{\sum_{k=0}^{3} 2^{-k} N_k}{\sum_{k=0}^{3} 2^{-k}}$$
where each $N_k$ is a cubic-zoom upsampling of a random grid of size $6 \cdot 2^k$. Each dot $i$ has position $(x_i, y_i) \sim \mathcal{U}([0,1]^2)$ perturbed by $\mathcal{N}(0, \sigma_j)$ jitter, and colour value $v_i = \text{clip}(F_{y_i, x_i} + \varepsilon_i,\; 0, 1)$ with $\varepsilon_i \sim \mathcal{U}(-0.08, 0.08)$.

**Logic in the Code**
All dot positions, jitter values, and colour-sample indices are computed in vectorised numpy arrays. The call `cmap(vals)` maps the entire array to RGBA in one step, avoiding a Python loop. Dot size $s_i = s_{\text{base}} \cdot u_i$ with $u_i \sim \mathcal{U}(0.6, 1.4)$ varies the apparent brushstroke scale. A single `ax.scatter` call renders all dots simultaneously, keeping the render fast even at 10 000 dots.

**Interesting Properties**
At low `n_dots` the individual coloured spots are clearly visible. Increasing `n_dots` causes the field to appear to merge into a continuous wash — exactly the optical integration Seurat sought. Increasing `dot_size` while lowering `n_dots` produces a Lichtenstein-style halftone grid. Jitter near zero creates a regular grid; high jitter produces a scattered, painterly look.


### How This Works — Cubist Portrait Filter

**Core Idea**
Low-poly (Delaunay-based) art decomposes an image into angular polygon facets, each filled with a flat colour sampled from an underlying field. This mirrors Cubism’s fragmentation of a subject into geometric planes viewed simultaneously from multiple angles. The mathematical foundation is the **Delaunay triangulation**, which constructs a mesh of non-overlapping triangles that maximises the minimum interior angle — producing compact, well-shaped facets.

**Mathematics**
Given $n$ interior points $\{\mathbf{p}_i\} \subset [0,1]^2$ plus 36 border-guard points on the canvas edges, the Delaunay triangulation partitions the convex hull into triangles $T_k$. Each triangle is coloured by sampling a noise field $F$ at its centroid:
$$\bar{\mathbf{p}}_k = \frac{1}{3}\sum_{j=1}^{3} \mathbf{p}_{k_j}, \qquad \text{color}_k = \text{cmap}\bigl(F[\bar{p}_{k,y},\; \bar{p}_{k,x}]\bigr)$$

**Logic in the Code**
`scipy.spatial.Delaunay` performs the triangulation in $O(n \log n)$. A $200 \times 200$ cubic-upsampled noise field (zoom factor 20 from a $10 \times 10$ seed) provides the smooth colour gradient. Border-guard points at canvas edges prevent large peripheral triangles that would otherwise appear where the interior point set is sparse. Each triangle is rendered as a `matplotlib.patches.Polygon` with a black edge controlled by `line_width`.

**Interesting Properties**
Decreasing `n_facets` to 20–40 produces bold geometric planes reminiscent of 1920s Cubist paintings. Increasing to 300+ makes the facets small enough that the image resolves toward a continuous gradient. Setting `line_width=0` removes all black outlines, dissolving the hard shard quality into a smooth mosaic effect that resembles stained glass without leading.


### How This Works — Abstract Expressionism Drip

**Core Idea**
Action painting (pioneered by Jackson Pollock in 1947–1950) moves paint through physical gesture — dripping, pouring, and splattering from above a canvas laid flat on the floor. Gravity and the paint’s viscosity govern form rather than deliberate brushwork. This renderer simulates paint drips as velocity-damped random walks with a downward gravitational bias.

**Mathematics**
Each drip is a discrete trajectory $\{(x_t, y_t)\}$ governed by:
$$v_x^{t+1} = 0.97\,(v_x^t + \xi_x), \qquad v_y^{t+1} = 0.97\,(v_y^t - g + \xi_y)$$
where $g$ is the `gravity` parameter and $\xi_x, \xi_y \sim \mathcal{U}(-0.008, 0.008)$ are random perturbations. The damping factor $0.97$ models viscous drag: the paint loses kinetic energy to friction as it spreads. The trajectory terminates when $x_t \notin [0.01, 0.99]$ or $y_t < 0.01$.

**Logic in the Code**
Each drip accumulates coordinates in a Python list until a boundary condition triggers, then is rendered in one `ax.plot` call with `solid_capstyle="round"` to avoid sharp angular discontinuities. A cluster of small `Circle` patches at the terminal point $(x_T, y_T)$ simulates the paint splatter formed when a drip impacts the canvas surface. Drip widths $w = w_{\text{base}} \cdot u$, $u \sim \mathcal{U}(0.5, 2.0)$, approximate different paint viscosities.

**Interesting Properties**
Setting `gravity=0` removes directional bias, making drips wander isotropically — producing a result closer to Cy Twombly’s undirected scribbling. Large `gravity` values (above 0.05) make all drips fall steeply and uniformly, resembling Yves Klein’s vertical drip works. The terminal splatter count $n_{\text{splat}} \sim \mathcal{U}(3, 12)$ replicates the characteristic paint-splash footprints visible in Pollock’s original large-format canvases.


---

## Section 4-D — 2D Game-Style Patterns (61–70)

Game-inspired visual algorithms: procedural generation, cellular automata, pathfinding, and retro rendering techniques.

### How This Works — Maze Generator & Solver

**Core Idea**
A perfect maze has exactly one path between any two cells — it is a spanning tree of the grid graph. The **recursive-backtracker** (randomised DFS) generates one by carving passages between unvisited neighbours chosen at random, producing long winding corridors with few dead ends. **BFS** then finds the unique shortest path from start to exit, highlighted in violet.

**Mathematics**
The grid is a graph $G=(V,E)$ where $V=\{(r,c):0\le r<R,\;0\le c<C\}$. DFS builds a random spanning tree by removing walls (edges) one at a time. The BFS shortest-path distance satisfies:
$$d(s,t)=\min\bigl\{|P|:P\text{ is a path from }s\text{ to }t\text{ in }T\bigr\}$$
Because $T$ is a tree this path is unique. Walls are stored in two boolean arrays — $\texttt{wall\_h}[R+1][C]$ (horizontal) and $\texttt{wall\_v}[R][C+1]$ (vertical) — making passage detection O(1).

**Logic in the Code**
1. Both wall arrays are initialised to *True* (all walls present).
2. An iterative DFS stack drives the backtracker; at each step a random unvisited neighbour is picked and the shared wall is removed.
3. BFS runs on the wall-free graph with a `prev` predecessor map; the solution path is traced backwards from the exit cell $(R-1,C-1)$.
4. Matplotlib draws walls as line segments; path cells are shaded violet; start (green) and end (red) are marked with **S** / **E**.

**Interesting Properties**
The backtracker favours long straight runs, making mazes that look hand-drawn. Prim's algorithm produces shorter, bushier corridors. Because the result is a spanning tree, BFS finds the same unique solution regardless of traversal order — there is no ambiguity about the correct path.

### How This Works — Cellular Automaton Life

**Core Idea**
Conway's Game of Life (1970) is a zero-player cellular automaton on a 2D grid. Each cell is alive or dead; at every generation all cells update simultaneously by simple neighbour-count rules. Despite its simplicity, Life is Turing-complete and produces emergent patterns — still-lifes, oscillators, gliders — that were never explicitly programmed.

**Mathematics**
Let $n_{r,c}$ be the number of live neighbours of cell $(r,c)$ (Moore neighbourhood, 8 cells). The update rule is:
$$\text{grid}_{r,c}^{(t+1)} = \begin{cases}1 & \text{if } n_{r,c}=3\text{ (birth)}\\1 & \text{if }\text{grid}_{r,c}^{(t)}=1\text{ and }n_{r,c}\in\{2,3\}\text{ (survival)}\\0 & \text{otherwise}\end{cases}$$
Neighbour counts use toroidal (wrapped) boundaries so edge patterns re-enter on the opposite side.

**Logic in the Code**
1. The initial grid is a Bernoulli sample: each cell independently alive with probability `density`.
2. A 3x3 ones-except-centre convolution kernel applied via `scipy.ndimage.convolve(..., mode='wrap')` counts all 8 neighbours simultaneously in $O(N^2)$ — no Python loop over cells.
3. Boolean masks `born` and `alive` implement the update rule without branching.
4. An `age` counter increments for every generation a cell stays alive; $\log(1+\text{age})$ drives the colourmap — young cells appear bright, long-lived cells darker.

**Interesting Properties**
Life undergoes a phase transition near initial density $p\approx0.37$: too sparse and patterns die quickly; too dense and overcrowding kills most cells. Around this critical density patterns persist longest and produce the richest oscillators and gliders. The density control lets you explore both sides of this transition.

### How This Works — Dungeon Room Placer

**Core Idea**
Procedural dungeon generation places rectangular rooms on a grid and connects them with corridors — the defining technique of roguelike games (Rogue 1980, Nethack, Dwarf Fortress). Every run produces a different but always-navigable floor plan.

**Mathematics**
Two axis-aligned rectangles $(x_1,y_1,w_1,h_1)$ and $(x_2,y_2,w_2,h_2)$ **overlap** iff:
$$x_1 < x_2+w_2 \;\wedge\; x_1+w_1 > x_2 \;\wedge\; y_1 < y_2+h_2 \;\wedge\; y_1+h_1 > y_2$$
A 1-cell margin is added so rooms never touch. L-shaped corridors connect room centres: first a horizontal run, then a vertical run to complete the connection.

**Logic in the Code**
1. Up to $10\times n\_rooms$ random $(x,y,w,h)$ proposals are tested; accepted rooms are stamped onto a NumPy grid (value 1 = floor).
2. Rooms are shuffled and connected pairwise with L-shaped corridors (value 2); only solid cells are overwritten to avoid erasing floor tiles.
3. A 3-channel RGB image maps rock to near-black, floor to warm stone, corridor to darker stone; gold pixels mark room centres.
4. `imshow(origin='upper')` places row 0 at the top, matching the dungeon coordinate convention.

**Interesting Properties**
Random placement degrades on small grids — the overlap test rejects too many proposals. Binary Space Partitioning (BSP) — recursively halving the grid — guarantees room placement but produces more regular, symmetric layouts. The gold centre markers reveal the order rooms were connected, tracing the minimum corridor network.

### How This Works — Retro Starfield

**Core Idea**
A 3D starfield simulates travelling through space, popularised by 1980s arcade games and screensavers. Stars are placed in 3D and projected onto 2D with perspective division; depth controls brightness and size, creating the illusion of forward motion.

**Mathematics**
Perspective projection maps a 3D point $(x, y, z)$ to screen coordinates:
$$p_x = \frac{x}{z}, \qquad p_y = \frac{y}{z}$$
Stars with small $z$ (close) project far from the origin and appear large and bright. The **warp** parameter $w\in[0,1]$ controls depth distribution via a power-law $z \sim \mathrm{Power}(1 + 4w)$, so higher warp skews stars toward $z\approx0$ — the hyperspace rush effect. Star size scales as $(1-z)^2\times 60$ and brightness as $(1-z)$.

**Logic in the Code**
1. $(x,y)\sim\mathcal{U}[-1,1]^2$ and $z\sim\mathrm{Power}(1+4w)$; stars outside view frustum $|p_x|,|p_y|>2$ are clipped after projection.
2. Near stars ($z<0.25$) receive radial streak lines from origin outward, simulating motion blur.
3. Star colours blend from blue-white (deep) to pure white (near) via per-channel linear interpolation.
4. A soft multi-ring glow at $(0,0)$ reinforces the vanishing-point focal effect.

**Interesting Properties**
At $w=0$ the depth distribution is uniform and the field looks static. As $w\to1$ nearly all stars are very close and streaks dominate, recreating the "jump to hyperspace" tunnel. Beyond $\approx2000$ stars, most visual density comes from the small near fraction — the distant majority is invisible due to their tiny projected size.

### How This Works — Breakout Brick Map

**Core Idea**
Breakout (Atari, 1976) arranges coloured bricks in a grid for a ball to destroy. This pattern reimagines the layout as a generative art canvas, applying four colouring schemes to the classic brick grid and adding bevel highlights for a retro plastic feel.

**Mathematics**
Four value functions $v_{r,c}\in[0,1]$ drive brick colour through the chosen palette colormap:

| Style | Formula |
|---|---|
| Row Gradient | $v_{r,c}=r/(R-1)$ |
| Checkerboard | $v_{r,c}=(r+c)\bmod 2$ |
| Radial | $v_{r,c}=\lVert(c-C/2,\,r-R/2)\rVert_2\,/\,\lVert(C/2,\,R/2)\rVert_2$ |
| Noise Art | $v_{r,c}\sim\mathcal{U}(0,1)$ |

All values are min-max normalised to $[0,1]$ before colormap lookup.

**Logic in the Code**
1. A $(R,C)$ float array is filled per style then normalised.
2. Each brick is a `FancyBboxPatch` with rounded corners; a semi-transparent white strip (height 22%, alpha 0.28) along the top simulates a bevelled plastic highlight.
3. A paddle (rounded rectangle) and ball (circle) below the brick area provide game context.
4. The Noise Art style optionally removes 15% of bricks at random for visual variety.

**Interesting Properties**
The radial style creates a bullseye that makes the grid centre appear to recede. The row-gradient style reproduces the visual hierarchy of the 1976 original — red bricks at top worth most points, yellow at bottom worth least. Combining Noise Art with a two-colour palette produces pixel-art mosaics.

### How This Works — Pac-Man Ghost Pathfinding

**Core Idea**
In Pac-Man (Namco, 1980) each ghost uses a different targeting strategy to chase the player. This pattern visualises **BFS shortest-path routing** on a procedurally generated maze: each ghost's coloured path traces the exact route it would take to reach Pac-Man, with ghost sprites coloured after the original arcade characters.

**Mathematics**
BFS on an unweighted graph $G=(V,E)$ finds the shortest path $P^*$ from source $s$ (ghost) to target $t$ (Pac-Man):
$$P^* = \arg\min_{P:\,s\to t}\,|P|$$
A FIFO queue expands cells in non-decreasing distance order. The predecessor map $\pi:V\to V$ records each cell's parent; the path is recovered by tracing backwards $P^* = (t,\pi(t),\pi(\pi(t)),\ldots,s)$.

**Logic in the Code**
1. An odd-dimension $N\times N$ grid is initialised to all-walls; recursive-backtracker DFS carves open passages, producing a perfect maze.
2. Pac-Man is placed at the open cell closest to the grid centre; ghosts are placed at random open cells at least $\lfloor N/3\rfloor$ Manhattan steps away.
3. BFS runs independently for each ghost, building its own `prev` dictionary and tracing back from Pac-Man.
4. Paths are drawn as semi-transparent polylines; ghost sprites combine `FancyBboxPatch` (body), `Circle` patches (eyes, fringe), and text labels; Pac-Man is a `Wedge` with a black eye.

**Interesting Properties**
Because the maze is a **perfect maze** (a spanning tree), there is exactly one path between any two open cells — BFS finds it in $O(N^2)$ time. In the real Pac-Man arcade game the maze contains cycles, making multiple shortest paths possible, and each ghost actually targets a different tile rather than Pac-Man's exact current position.

### How This Works — Platformer Terrain Gen

**Core Idea**
Procedural platformer terrain mimics the hand-crafted levels of games like Super Mario Bros by generating terrain algorithmically. A 1D fractal Brownian motion (fBm) curve shapes the ground, while random floating platforms, coins, and a player sprite complete the scene across three visual themes: grassy, cave, and snow.

**Mathematics**
The ground height $h(x)$ is a 1D fBm sum of $k$ octaves:
$$h(x) = \frac{\sum_{k=0}^{K-1} 0.5^k \cdot \text{interp}_{2^k}(x)}{\sum_{k=0}^{K-1} 0.5^k}$$
where $\text{interp}_{2^k}$ is a cubic-spline interpolation of a random control-point grid at frequency $2^k$. The result is normalised to $[0.08, 0.46]$ so the terrain stays within the lower canvas half. Platform positions $(p_x, p_y, p_w)$ are drawn uniformly, with a height rejection test ensuring they sit above the terrain surface.

**Logic in the Code**
1. `fbm1d(n)` builds the terrain: for each octave, a sparse random array is cubic-interpolated to length $n$ via `scipy.interpolate.interp1d`, accumulated with halving amplitude, then normalised.
2. The terrain is rendered as a filled polygon (`ax.fill`) for the dirt body, plus a per-segment quad strip for the coloured grass cap.
3. Platforms are `FancyBboxPatch` + `Rectangle` top-stripe; coins are `Circle` with highlight dot; the player sprite uses three rectangle/circle patches for body, head, and cap.
4. A sky gradient is painted as 20 thin horizontal bands interpolated between two theme colours.

**Interesting Properties**
fBm terrain is self-similar across scales — zooming into any section reveals the same statistical roughness as the whole. The `style` control switches only colours and palette, demonstrating how the same generative geometry reads as completely different environments through colouring alone.

### How This Works — Bullet Hell Pattern

**Core Idea**
Bullet hell (danmaku) shooters challenge players to navigate dense, patterned projectile fields. The visual complexity emerges from simple angular arithmetic: by placing bullets at regular angular intervals across multiple expanding rings, and varying the offset between rings, five distinct pattern families appear — circular bursts, spirals, fans, star crosses, and scatter.

**Mathematics**
For a circular burst with $N$ bullets per ring and ring index $k$, bullet $b$ in ring $k$ sits at polar coordinates:
$$r_k = 0.15 + 0.18k, \qquad \theta_{k,b} = \frac{2\pi b}{N} + k\cdot\delta$$
where $\delta$ is a per-ring phase offset that staggers the rings. The double-spiral uses two arms offset by $\pi$ with a continuous radius ramp: $r = 0.08 + 0.08s$ for step $s$. The aimed fan distributes $N$ angles uniformly across a spread arc centred on the player bearing $\theta_\text{player} = \pi/2$.

**Logic in the Code**
1. A list of `(x, y, t)` tuples is built for all bullets; $t\in[0,1]$ encodes age (ring index / max rings) for colourmap lookup.
2. Each bullet is drawn as two overlapping scatter points: a small bright core and a larger low-alpha outer glow — a cheap but effective bloom effect.
3. Faint concentric guide rings at each bullet radius are drawn at alpha 0.07 to reveal the underlying pattern geometry.
4. A boss sprite (three concentric circles) sits at the origin; a player triangle marker sits above at $y=1.10$.

**Interesting Properties**
Bullet hell patterns are isomorphic to Lissajous figures and epicycloid curves at the limit of continuous angular density. The double-spiral at high bullet count converges to the Archimedean spiral $r=a\theta$, while the cross-star produces the 8-petalled rose curve $r=\cos(4\theta)$.

### How This Works — Card Suit Patterns

**Core Idea**
The four card suits — heart, diamond, club, spade — are among the most recognised symbols in Western culture, appearing in playing cards since 15th-century France. This pattern renders all four using parametric and geometric formulas, then presents them in three layout modes: a large 2×2 grid, a repeating tile mosaic, and a stylised ace-of-hearts card face.

**Mathematics**
The **heart** uses the classic parametric curve:
$$x(t) = 16\sin^3 t, \qquad y(t) = 13\cos t - 5\cos 2t - 2\cos 3t - \cos 4t$$
normalised to $[-1,1]^2$. The **diamond** is a rhombus with vertices at $(\pm s, 0)$ and $(0, \pm s)$. The **club** is three equal circles of radius $0.48s$ arranged in an equilateral triangle plus a rectangular stem. The **spade** is the heart curve with $y\to-y$ (y-flipped) and an added rectangular stem.

**Logic in the Code**
1. `_heart`, `_diamond`, `_club`, `_spade` are static methods returning polygon vertices or patch lists.
2. `draw_suit` dispatches to the correct generator and adds `ax.fill` (heart, diamond, spade) or `Circle`+`FancyBboxPatch` patches (club).
3. Layout 0 places one suit per quadrant at large scale with a name label below.
4. Layout 1 tiles a 4×4 grid cycling through the four suits; layout 2 builds a card face with a central heart, four corner pips, and rank labels with 180° rotations for the bottom pair.

**Interesting Properties**
The parametric heart formula is a trigonometric polynomial — a finite Fourier series in $t$. Adding higher-frequency cosine terms $(-2\cos 3t - \cos 4t)$ sharpens the bottom cusp and deepens the top indentation. Removing those terms produces a smooth oval instead of a heart shape.

### How This Works — Pixel Flag Generator

**Core Idea**
National flags follow a small set of recurring geometric grammars: horizontal/vertical stripes, crosses, diagonal divisions, and central emblems. This pattern procedurally generates pixel-art flags using six of those grammars, mapping a chosen palette colourmap to $n$ discrete flag colours and rasterising the result onto a $90\times60$ pixel canvas.

**Mathematics**
- **Horizontal stripes:** cell row $r$ maps to stripe $\lfloor r\cdot n_s / H \rfloor \bmod n_c$.
- **Diagonal bicolour:** pixel $(r,c)$ is colour 0 if $c/W < r/H$ (below the main diagonal), else colour 1.
- **Nordic cross:** two overlapping bars at positions $h_1:h_2$ (horizontal) and $v_1:v_2$ (vertical).
- **5-pointed star rasterisation:** for each pixel, $\text{dist} = \sqrt{dx^2+dy^2}$ is compared to an angle-dependent boundary $r(\theta) = r_{\text{inner}} + (r_{\text{outer}} - r_{\text{inner}})\cdot|\sin(\pi\cdot\theta_{\text{sector}})|$.

**Logic in the Code**
1. $n_c$ colours are sampled uniformly from the chosen palette colormap and stored as RGB float triples.
2. Each design fills a $(H, W, 3)$ NumPy array with colour values using vectorised or simple nested loops.
3. The finished flag is up-scaled 5× via `np.repeat` along both axes for pixel-perfect display, then shown with `imshow(interpolation='nearest')` to preserve sharp pixel edges.
4. A thin rectangle border is drawn over the scaled image to simulate a flag frame.

**Interesting Properties**
The Nordic cross grammar (offset vertical bar) is shared by the flags of Denmark, Norway, Sweden, Finland, and Iceland — demonstrating how a single geometric rule can encode an entire cultural region. The noise-mosaic design generates what appear to be patchwork quilts or abstract stained glass when colour count is set to 2.

### How This Works — Rotating DNA Helix

**Core Idea**
The DNA double helix is modelled as two helical strands wound around a shared axis,
connected by colour-coded base-pair rungs. The 3D backbone is rendered as a gradient
scatter plot using `mpl_toolkits.mplot3d`, giving the illusion of a ribbon twisting in space.

**Mathematics**
Each strand is a helix: $x_1 = r\cos t$, $y_1 = r\sin t$, $z = t / 2\pi$ with the second
strand offset by $\pi$ radians: $x_2 = r\cos(t+\pi)$, $y_2 = r\sin(t+\pi)$.
Base pairs are straight line segments connecting $(x_1, y_1, z)$ to $(x_2, y_2, z)$
sampled uniformly along the helix.

**Logic in Code**
A `numpy` array of $t$ values produces the helix coordinates. Colour is mapped via a
`LinearSegmentedColormap` applied to normalised $z$-height. The four nucleotide pair
colours (A–T, C–G proxies) are cycled across rung indices. `ax.scatter` with `depthshade=True`
provides automatic depth cuing.

**Interesting Properties**
The human genome fits ~3 billion base pairs into 2 m of total helix length by coiling
at multiple scales (supercoiling). A single full turn of the helix (one pitch) spans
~3.4 nm and contains exactly 10 base pairs in B-DNA.


### How This Works — Klein Bottle Surface

**Core Idea**
The Klein bottle is a non-orientable surface with no boundary — like a Möbius strip but
closed. It cannot be embedded in 3D without self-intersection, so what we render is an
*immersion* (a self-intersecting 3D projection of the true 4D object).

**Mathematics**
The figure-8 immersion uses parameters $u, v \in [0, 2\pi)$:
$$X = \left(2 + \cos\tfrac{u}{2}\sin v - \sin\tfrac{u}{2}\sin 2v\right)\cos u$$
$$Y = \left(2 + \cos\tfrac{u}{2}\sin v - \sin\tfrac{u}{2}\sin 2v\right)\sin u$$
$$Z = \sin\tfrac{u}{2}\sin v + \cos\tfrac{u}{2}\sin 2v$$

**Logic in Code**
`np.meshgrid(u, v)` creates the parameter grid. The three component arrays are computed
vectorised and passed to `ax.plot_surface` with `facecolors=cmap(Zn)` where $Z_n$ is
the normalised $Z$ value, producing a height-based colouring. `alpha < 1` allows the
self-intersection region to be visible.

**Interesting Properties**
The Klein bottle has Euler characteristic 0 (same as a torus), yet it is non-orientable.
Inside and outside are meaningless concepts for it — a path along the surface eventually
returns to the starting point from the opposite side.


### How This Works — Möbius Strip

**Core Idea**
The Möbius strip is the simplest non-orientable surface: a rectangle with one half-twist
before its ends are joined. It has only one side and one edge, making it a classic object
in topology.

**Mathematics**
Standard parametric form with $u \in [0, 2\pi)$ and $v \in [-w, w]$:
$$x = \left(1 + v\cos\tfrac{u}{2}\right)\cos u, \quad
  y = \left(1 + v\cos\tfrac{u}{2}\right)\sin u, \quad
  z = v\sin\tfrac{u}{2}$$
The `twist` parameter replaces the half-angle $\tfrac{u}{2}$ with $\tfrac{\text{twist} \cdot u}{2}$,
producing strips with 1, 3, 5… half-twists (odd = non-orientable, even = cylinder).

**Logic in Code**
A `meshgrid` over $(u, v)$ yields the surface arrays. `ax.plot_surface` renders the mesh
with azimuthal colouring $U/(2\pi)$ mapped through the chosen palette. Semi-transparency
(`alpha`) lets the viewer see through the single surface to appreciate the twist.

**Interesting Properties**
Cutting a Möbius strip down the centre does not split it — it produces a single strip
with two full twists and twice the length. Cutting at one-third produces two interlocked
loops of different sizes.


### How This Works — Torus Knot

**Core Idea**
A torus knot $T(p, q)$ is a knot that winds $p$ times around the longitude of a torus
and $q$ times around its meridian without self-intersecting. The resulting space curve
has a striking helical symmetry.

**Mathematics**
Using a single parameter $t \in [0, 2\pi)$:
$$x = (R + r\cos qt)\cos pt, \quad
  y = (R + r\cos qt)\sin pt, \quad
  z = r\sin qt$$
where $R$ is the torus major radius, $r$ the minor radius.
$T(2,3)$ is the trefoil; $T(3,4)$ is the $(3,4)$-torus knot (8 crossings).

**Logic in Code**
A 1D `numpy` array of $t$ values produces the curve. The path is split into consecutive
segments and rendered with `Line3DCollection`, each segment coloured by its position
along the curve via the palette's colormap. This gives a smooth rainbow gradient around
the knot.

**Interesting Properties**
$\gcd(p, q) = 1$ is required for the curve to be a knot rather than a link. When
$p = 2, q = 3$ the trefoil knot results — it is the simplest non-trivial knot and
cannot be untied without cutting the strand.


### How This Works — Gyroid Surface

**Core Idea**
The gyroid is a triply periodic minimal surface (TPMS) discovered by Alan Schoen in 1970.
It has zero mean curvature everywhere, no straight lines, and no mirror planes, yet it
tiles 3D space with perfect regularity. It appears in butterfly wing nanostructures and
block copolymer self-assembly.

**Mathematics**
Defined implicitly by:
$$\cos x \sin y + \cos y \sin z + \cos z \sin x = 0$$
in a periodic domain. Points near the zero level set (within `threshold`) are selected
from a 3D grid and plotted as a scatter cloud.

**Logic in Code**
`np.meshgrid` over $[-2\pi, 2\pi]^3$ builds a dense grid. The implicit function $F$
is evaluated vectorised. A boolean mask `|F| < \text{threshold}` extracts the surface
shell as a point cloud. `ax.scatter` with height-based colouring reveals the surface's
complex yet periodic geometry.

**Interesting Properties**
The gyroid separates space into two congruent, non-intersecting labyrinthine regions.
Unlike most TPMS, it has no straight lines or planar symmetry elements, making it
chiral — it exists in left- and right-handed forms.


### How This Works — Romanesco Broccoli

**Core Idea**
Romanesco broccoli is a natural fractal: each conical bud is itself composed of smaller
identical buds arranged in the same spiral pattern, recursively. The overall head follows
a Fibonacci spiral (golden-angle phyllotaxis), producing a logarithmic spiral arrangement
of self-similar cones.

**Mathematics**
Golden-angle phyllotaxis places bud $i$ at angle $i \cdot \varphi_g$ where
$\varphi_g = 2\pi(1 - 1/\phi^2) \approx 2.400$ rad ($\phi$ = golden ratio).
Radial distance grows as $\rho = 0.8\sqrt{i/N}$ and height as $z = 1.5\,i/N$,
producing the characteristic conical spiral. Sub-bud clusters at each main bud
simulate the recursive self-similarity.

**Logic in Code**
A flat loop over $N$ bud positions computes $(r, \theta, z)$ in conical-spiral coordinates.
For each main bud, a small cluster of sub-buds is placed via a nested golden-angle loop
scaled by bud size. All points are accumulated in flat arrays and rendered with
`ax.scatter`, colour-mapped by hierarchy level.

**Interesting Properties**
The number of spirals visible in each direction always forms consecutive Fibonacci numbers
(e.g.\ 8 and 13, or 13 and 21), a property shared by sunflower seed heads and pine cones.
This emerges from the irrational golden angle naturally packing points with maximum
angular separation.


### How This Works — Icosphere Subdivisions

**Core Idea**
An icosphere is a sphere approximation built by iteratively subdividing the faces of a regular icosahedron. Starting from only 20 equilateral triangles, each subdivision replaces every triangle with four smaller ones and projects all new vertices back onto the unit sphere. The result converges toward a perfect sphere while maintaining a fully triangulated mesh — making it the preferred sphere primitive in 3D graphics, physics engines, and geodesic architecture.

**Mathematics**
The base icosahedron uses 12 vertices constructed from the golden ratio $\phi = (1+\sqrt{5})/2$:

$$V = \frac{1}{\|(\pm 1, \pm\phi, 0)\|}\left\{(\pm 1, \pm\phi, 0),\; (0, \pm 1, \pm\phi),\; (\pm\phi, 0, \pm 1)\right\}$$

Each subdivision replaces face $(A, B, C)$ with four faces by inserting midpoints $M_{AB} = \frac{A+B}{\|A+B\|}$, $M_{BC}$, $M_{CA}$ — each normalised to radius 1. After $n$ subdivisions the face count is $F = 20 \cdot 4^n$.

**Logic in the Code**
1. The 12 base vertices are normalised to the unit sphere and stored in a mutable list; the 20 base faces are index triples.
2. `subdivide()` processes each face: for each edge $(i,j)$ it computes the normalised midpoint exactly once (cached by sorted pair $(\min,\max)$), appends it to the vertex list, and replaces the face with four children.
3. This is repeated `subdivisions` times. At Low/Medium/High resolution the depth is 2/3/4 (320/1280/5120 faces).
4. Face centroids are used to map a height-based colour value through the chosen palette, passed to `Poly3DCollection`.

**Interesting Properties**
Every icosphere vertex lies exactly on the unit sphere, but edge lengths are not all equal — unlike the base icosahedron. The `subdivisions` slider makes this explicit: at 0 the platonic solid is clearly visible, while at 4 the mesh is nearly indistinguishable from a smooth sphere. Geodesic domes (pattern \#83) apply the same subdivision logic to a hemisphere.

### How This Works — Trefoil Knot

**Core Idea**
The trefoil knot is the simplest non-trivial knot — a closed curve in 3D space that cannot be continuously deformed into a plain loop without cutting. It takes its name from the three-leaf clover shape it resembles when projected onto a plane. In knot theory it is denoted $3_1$ and is chiral: it is not equivalent to its mirror image.

**Mathematics**
The standard parametric embedding is:

$$x(t) = \sin t + 2\sin 2t, \quad y(t) = \cos t - 2\cos 2t, \quad z(t) = -\sin 3t, \quad t \in [0,\, 2\pi]$$

The curve closes after one period $2\pi$. Equivalently, the trefoil is the torus knot $T(2,3)$: a curve that winds $p=2$ times around the longitude and $q=3$ times around the meridian of a torus — making it a special case of the torus knot family already explored in pattern \#74.

**Logic in the Code**
1. A parameter array of $N$ equally spaced values over $[0, 2\pi]$ is generated and the three coordinate arrays are computed vectorially.
2. Consecutive point pairs are assembled into a `Line3DCollection` of $N-1$ segments; each segment is assigned a colour by linearly interpolating the palette across $[0, 1]$, producing a smooth gradient along the knot.
3. `auto_scale_xyz` fits the axes to the curve's bounding box automatically — no manual limits required.
4. `linewidth` is exposed as a control; increasing it emphasises the over/under crossings that define the topology of the knot.

**Interesting Properties**
The trefoil exhibits three-fold rotational symmetry ($C_3$), clearly visible from directly above. Because the trefoil is chiral, the left-handed and right-handed versions are topologically distinct — a property with direct biological relevance: DNA molecules sometimes form trefoil knots during replication, and the handedness of the knot affects how enzymes interact with it.

### How This Works — Seashell Surface

**Core Idea**
Mollusc shells grow following a logarithmic spiral in which each revolution of the coiling angle produces a cross-section proportionally larger than the last. This exponential self-similar growth is captured by a single parametric surface that wraps a circular tube around an expanding helicospiral — recreating the mathematics of the nautilus, cone snail, and ammonite in a few lines of numpy.

**Mathematics**
Two angles parametrise the surface. The coiling angle $v \in [0,\, 2\pi n]$ traces the spiral; the cross-section angle $u \in [0,\, 2\pi]$ sweeps the tube. The coil radius and tube radius grow exponentially with $v$:

$$R(v) = e^{b v}, \qquad r_t = eta\, R(v)$$

$$X = (R + r_t \cos u)\cos v, \quad Y = (R + r_t \cos u)\sin v, \quad Z = r_t \sin u + c\,v$$

Here $b$ is the `growth_rate` (controls tightness of coiling), $eta$ is `tube_scale` (proportional thickness), and $c = 0.4$ controls vertical rise per radian.

**Logic in the Code**
1. A 2D parameter grid $(V, U)$ is constructed via `meshgrid`, with $v$ spanning $[0, 2\pi \cdot n\_turns]$ and $u$ spanning $[0, 2\pi]$.
2. $R(V)$ is computed element-wise as `exp(growth_rate * V)`; `r = tube_scale * R` gives the tube radius.
3. All three coordinate arrays are evaluated with numpy broadcasting — no Python loops.
4. The surface height $Z$ is normalised to $[0, 1]$ and mapped through the chosen palette, giving a colour gradient from the closed tip to the open mouth.
5. `plot_surface` renders with `shade=True` for directional lighting.

**Interesting Properties**
The logarithmic spiral is the unique spiral self-similar under scaling: rotating by $\Delta v$ is equivalent to scaling by $e^{b \Delta v}$. Shells of this type are called *equiangular* — the tangent to the spiral makes a constant angle with any radial line. `growth_rate` directly controls this angle: small values give tight coils (ammonite), large values give open coils (nautilus). Setting `tube_scale` proportional to $R$ ensures the tube cross-section also scales self-similarly.


### How This Works — Hyperboloid of Revolution

**Core Idea**
A hyperboloid of one sheet is the quadric surface swept by rotating a hyperbola around its conjugate axis — producing the characteristic waist-narrowed shape of nuclear cooling towers and modernist lattice architecture. It is remarkable for being simultaneously a surface of revolution and a *doubly ruled surface*: two distinct families of perfectly straight lines lie entirely on it.

**Mathematics**
The one-sheeted hyperboloid satisfies:

$$\frac{x^2}{a^2} + \frac{y^2}{a^2} - \frac{z^2}{c^2} = 1$$

Its parametric form is:

$$x = a\cosh(t)\cos u,\quad y = a\cosh(t)\sin u,\quad z = c\sinh(t), \qquad t\in\mathbb{R},\; u\in[0,2\pi]$$

Verification uses the identity $\cosh^2 t - \sinh^2 t = 1$. The waist (minimum circle) occurs at $t=0$ with radius $a$. One family of ruling lines passes through $(a\cos\theta,\, a\sin\theta,\, 0)$ in direction $(-a\sin\theta,\, a\cos\theta,\, c)$ for each angle $\theta$.

**Logic in the Code**
1. A meshgrid over $t \in [-t\_range,\, t\_range]$ and $u \in [0,\, 2\pi]$ is created; $X$, $Y$, $Z$ are computed vectorially.
2. The surface is rendered with `plot_surface` coloured by normalised height $Z$.
3. When `show_lines` is enabled, 16 ruling lines are drawn as white semi-transparent overlays. Each is parametrised by scalar $s$ along the ruling direction, evaluated at 60 points via numpy, and plotted as a simple `ax.plot`.

**Interesting Properties**
Despite appearing curved, each ruling line is geometrically straight — architects exploit this in concrete cooling towers and hyperbolic lattice shells, since straight beams are cheaper to fabricate than curved ones. Adjusting `a` (waist radius) and `c_scale` (axial stretch) morphs the shape between a near-cylinder and a sharply pinched hourglass. Setting `a` very large relative to `c_scale` approximates a cylinder; letting `c_scale` dominate emphasises the hyperbolic profile.

### How This Works — Parametric Vase

**Core Idea**
Any surface of revolution is generated by rotating a planar profile curve $r(z)$ around the vertical axis. By choosing different profile functions — sinusoidal, Gaussian, polynomial — an infinite family of vessel shapes emerges. This pattern demonstrates four distinct ceramic profiles, each produced by a compact mathematical function revolved through $[0, 2\pi]$.

**Mathematics**
The parametric surface is:

$$X(u, z) = r(z)\cos u,\quad Y(u, z) = r(z)\sin u,\quad Z(u, z) = z, \qquad u\in[0,2\pi],\; z\in[0,1]$$

Each style defines a different profile $r(z)$:
- **Classic amphora** (0): $r = 0.15 + 0.55\sin(\pi z) + 0.12\sin(2\pi z)$, footed at base
- **Chinese vase** (1): $r = 0.4\,e^{-8(z-0.65)^2} + 0.1 + 0.25z(1-z)$ — Gaussian shoulders
- **Rippled cylinder** (2): $r = 0.3 + 0.1\sin(6\pi z) + 0.2\sin(\pi z)$ — high-frequency modulation
- **Round bulge** (3): $r = 0.15 + 0.55\sin(\pi z^{0.7})$ — asymmetric sine for forward-leaning belly

**Logic in the Code**
1. The 1D profile $r(z)$ is evaluated on $n\_z$ height samples using the formula for the selected style.
2. A 2D grid $(Z	ext{2d},\, U	ext{2d})$ is meshed; `np.interp` broadcasts the 1D profile onto the full grid.
3. $X$ and $Y$ are computed vectorially and passed to `plot_surface` with height-based colour normalisation.
4. The `style` integer is taken modulo 4 so the slider wraps around all four shapes.

**Interesting Properties**
The four profiles are short mathematical descriptions of real historical ceramic traditions — the amphora (ancient Greece), the Mei Ping vase (Song dynasty China), modernist studio pottery, and the round belly of Ming-era porcelain. Adding more Fourier terms to $r(z)$ — i.e., more $\sin(k\pi z)$ components — would allow arbitrary profile shapes via Fourier synthesis, demonstrating that classical vessel design can be encoded as a Fourier series in a single variable.

### How This Works — Crystal Lattice

**Core Idea**
Crystal lattices are the periodic arrangements of atoms in solid materials. The geometry of the lattice — how atoms are stacked and how they bond — determines physical properties such as density, electrical conductivity, and mechanical strength. This pattern renders four canonical crystal structures as 3D scatter plots with bond lines connecting nearest-neighbour atom pairs.

**Mathematics**
Each crystal structure is defined by lattice vectors $\mathbf{a}_1,\mathbf{a}_2,\mathbf{a}_3$ and a basis (motif) of fractional positions $\mathbf{d}_\alpha$ within the unit cell. The full supercell is:

$$\mathbf{R}_{ijk\alpha} = i\,\mathbf{a}_1 + j\,\mathbf{a}_2 + k\,\mathbf{a}_3 + d_{\alpha,1}\mathbf{a}_1 + d_{\alpha,2}\mathbf{a}_2 + d_{\alpha,3}\mathbf{a}_3$$

The four structures and their nearest-neighbour distances (for unit cell $a=1$):
- **Simple Cubic (SC):** one basis atom; NN distance $1$.
- **Body-Centred Cubic (BCC):** basis $\{(0,0,0),\,(\tfrac{1}{2},\tfrac{1}{2},\tfrac{1}{2})\}$; NN $= \tfrac{\sqrt{3}}{2}$.
- **Face-Centred Cubic (FCC):** 4-atom basis including all face-centre positions; NN $= \tfrac{\sqrt{2}}{2}$.
- **Diamond Cubic:** FCC + 4 tetrahedral interstitials at $(\tfrac{1}{4},\tfrac{1}{4},\tfrac{1}{4})$ etc.; NN $= \tfrac{\sqrt{3}}{4}$.

**Logic in the Code**
1. Atom positions are built by iterating over an $n \times n \times n$ supercell grid and adding each basis atom transformed to Cartesian coordinates.
2. The lattice is centred by subtracting the mean position.
3. When `show_bonds` is enabled, `scipy.spatial.cKDTree.query_pairs` efficiently finds all nearest-neighbour pairs within the cutoff distance.
4. Atoms are coloured by height via the chosen palette; `ax.scatter` renders them with `depthshade=True` for distance-based shading.

**Interesting Properties**
The FCC packing fraction is $\pi/(3\sqrt{2}) \approx 74\%$ \u2014 the theoretical maximum for equal spheres, proved by the Kepler conjecture in 2005. BCC packs at $68\%$ and SC at only $52\%$. Diamond's open tetrahedral bonding (packing $\approx 34\%$) gives silicon its semiconductor properties by leaving room for the directional $sp^3$ orbitals that form covalent bonds.

### How This Works — Geodesic Dome

**Core Idea**
A geodesic dome is a hemispherical structure made entirely of triangular struts, constructed by subdividing the faces of a regular icosahedron projected onto a sphere and discarding the lower hemisphere. Popularised by Buckminster Fuller in the 1950s, geodesic domes are structurally extremely efficient: the triangular tessellation distributes loads across all members, allowing thin material to enclose large volumes with no internal supports.

**Mathematics**
The construction uses identical subdivision logic to the icosphere (pattern \#77). After $f$ frequency subdivisions, the upper hemisphere retains faces whose centroid satisfies $z > -arepsilon$. The key metric is the *frequency* $f$: each icosahedron edge is divided into $f$ equal segments, giving approximately $10f^2$ triangular faces in the full sphere and $pprox 5f^2$ in the dome.

The strut lengths in a real geodesic dome are not all equal — they fall into a small number of distinct classes (typically 3 for frequency-2), whose lengths can be derived from the spherical trigonometry of the original icosahedron face.

**Logic in the Code**
1. An icosphere is built by the same vertex-and-face subdivision used in pattern \#77, to depth `frequency`.
2. Faces with centroid $z \leq -0.15$ are discarded, cutting the dome near the equator.
3. `Poly3DCollection` renders the triangular panels with semi-transparent white edges (the *struts*), coloured from cool blue (base ring) to warm (apex) via the chosen palette.
4. When `show_base` is enabled, a white ring is drawn at $z = -0.15$ to represent the dome's ground ring.

**Interesting Properties**
The ratio of enclosed volume to surface area grows with dome size — making large geodesic domes more material-efficient than small ones. The 12 pentagonal panels that appear in sufficiently subdivided domes (corresponding to the original icosahedron vertices) are the topological basis of buckminsterfullerene $	ext{C}_{60}$, which is effectively a geodesic sphere at molecular scale. This connection between architectural geometry and chemistry earned the molecule its name.

### How This Works — Calabi-Yau Manifold Slice

**Core Idea**
A Calabi-Yau manifold is a complex algebraic variety central to string theory, where it compactifies the six extra spatial dimensions into an imperceptibly small geometric shape. The image shows a 2D real projection of the algebraic surface $z_1^n + z_2^n = 1$ in $\mathbb{C}^2$; for $n = 5$ this is a slice of the quintic threefold studied extensively by Andrew Hanson in the 1990s.

**Mathematics**
The defining constraint is
$$z_1^n + z_2^n = 1, \quad z_1, z_2 \in \mathbb{C}.$$
Parameterise with $\rho \in [0,1]$, $\alpha \in [0, 2\pi/n]$, and branch indices $k, j \in \{0,\ldots,n-1\}$:
$$z_1^{(k)} = \rho\,e^{i(\alpha + 2\pi k/n)}, \qquad z_2^{(j)} = |1 - z_1^n|^{1/n}\,e^{i\!\left(\tfrac{\arg(1-z_1^n)}{n} + \tfrac{2\pi j}{n}\right)}.$$
Because $z_1^n = \rho^n e^{in\alpha}$ (the branch index $k$ cancels), one confirms $z_1^n + z_2^n = 1$ for all $(k,j)$. The 2D projection maps each pair to $(\operatorname{Re} z_1^{(k)},\, \operatorname{Re} z_2^{(j)})$, with colour encoding $\operatorname{Im} z_1 + \operatorname{Im} z_2$.

**Logic in the Code**
1. A 2D grid over $(\rho, \alpha)$ is built with `np.meshgrid`; `z1_n` and `z2_n` are computed with numpy complex arithmetic in one vectorised step.
2. For each of the $n^2$ branch pairs the phases $\theta_1 = \alpha + 2\pi k/n$ and $\theta_2 = \arg(z_2^n)/n + 2\pi j/n$ are computed; real and imaginary parts follow from `cos`/`sin`.
3. All $n^2$ scatter arrays are concatenated and rendered in one `ax.scatter` call with `rasterized=True` for performance at high resolution.

**Interesting Properties**
For $n=2$ the four patches reduce to a familiar conic; as $n$ grows the $n^2$ patches multiply, producing the intricate lattice symmetry visible at $n=5$. The image has exact $n$-fold rotational symmetry in both $\operatorname{Re}(z_1)$ and $\operatorname{Re}(z_2)$, reflecting the $\mathbb{Z}/n\mathbb{Z}$ symmetry of the equation under $z_1 \mapsto e^{2\pi i/n} z_1$.

### How This Works — Soap Bubble Cluster

**Core Idea**
Soap films obey Plateau’s laws: they form surfaces of constant mean curvature (spheres for isolated bubbles) and meet three-at-a-time at 120° contact angles. This render approximates a cluster of spherical soap bubbles with iridescent thin-film colouring — the source of soap’s rainbow sheen.

**Mathematics**
An isolated bubble is a sphere because the sphere minimises surface area for fixed enclosed volume, satisfying the Laplace pressure equation:
$$\Delta P = P_{\text{inside}} - P_{\text{outside}} = \frac{4\gamma}{R}$$
where $\gamma$ is surface tension and $R$ is bubble radius. Thin-film interference produces colour when the film thickness $d$ and light wavelength $\lambda$ satisfy $2nd = m\lambda$ ($n$ = refractive index, $m \in \mathbb{Z}$). Because soap film thickness varies continuously, all wavelengths interfere simultaneously, producing the full iridescent spectrum.

**Logic in the Code**
1. Bubble radii are drawn from a uniform distribution; centres are placed greedily — a new centre is accepted only if its Euclidean distance to every existing centre exceeds 92\% of the sum of their radii, giving a realistic touching cluster.
2. A unit-sphere mesh $(U, V)$ is pre-computed once; each bubble offsets and scales it, then calls `ax.plot_surface` with semi-transparent HSV colour.
3. Hue cycles uniformly across bubbles (offset by a random `hue_base`) and is converted to RGB via `colorsys.hsv_to_rgb`, approximating the shifting interference colours of real soap films.
4. A small white scatter point placed just inside the surface at a fixed direction simulates the specular highlight seen on every real soap bubble.

**Interesting Properties**
Real soap clusters obey Plateau’s fourth law: four bubble walls always meet at a Steiner point at the tetrahedral angle $\arccos(-1/3) \approx 109.47^\circ$. This is why soap foam cells resemble Voronoi cells — the surface-tension minimisation drives topology toward the Kelvin structure, long conjectured (and later disproved by Weaire and Phelan in 1994) to be the most efficient space-filling foam.

### How This Works — Neural Mesh Sculpture

**Core Idea**
A neural mesh sculpture abstracts the architecture of a deep feedforward network into a 3D geometric object: parallel planes of neurons connected by synaptic edges between adjacent layers. The piece visualises directed information flow — input on the left, output on the right — as a luminous sculptural structure.

**Mathematics**
In a feedforward network with $L$ layers, neuron $i$ in layer $\ell$ connects to neuron $j$ in layer $\ell+1$ with weight $w_{ij}$, propagating:
$$a_j^{(\ell+1)} = \sigma\!\left(\sum_i w_{ij}\,a_i^{(\ell)} + b_j\right)$$
where $\sigma$ is a non-linear activation. The sculpture encodes topology only — no weights are stored. A connection is drawn between neurons $i$ and $j$ in adjacent layers when their 3D Euclidean distance $\|p_i - p_j\|$ is below `conn_threshold`. Increasing this parameter grows the graph from sparse to nearly fully connected ($K_{m,n}$, the complete bipartite graph between adjacent layers).

**Logic in the Code**
1. For each layer $\ell$, neuron positions are sampled uniformly in the $y$-$z$ plane at $x = 2\ell/(L-1) - 1$, placing all layers between $x = -1$ and $x = 1$.
2. A `scipy.spatial.cKDTree` is built for each target layer; for each source neuron a $k$-nearest-neighbour query finds candidates within `conn_threshold`, accumulating accepted edges into lists.
3. All edges are rendered in one `Line3DCollection` call — far more efficient than individual `ax.plot` calls. Edge alpha is set to 0.30 to create a depth-haze effect without depth-sorting.
4. Neuron scatter sizes are modulated: input and output layer neurons are rendered at 80\,pt$^2$, hidden neurons at 35\,pt$^2$, emphasising the network boundaries.

**Interesting Properties**
The `conn_threshold` slider crosses a percolation threshold: below it the graph is sparse and forest-like; above it a giant connected component spans all layers simultaneously. This transition — visible as a sudden shift from disconnected islands to a web of light — is the visual analogue of the percolation phase transition studied in network theory.

### How This Works — Twisted Prism Tower

**Core Idea**
A twisted prism tower stacks regular $n$-sided polygon floors, each rotated by an incremental angle as it rises, producing the helical facade seen in landmarks like the Turning Torso in Malmö and the Shanghai Tower. The geometry is a ruled surface: every side face is bounded by two straight edges.

**Mathematics**
For a tower with $N_f$ floors and total twist $\Theta$, the $k$-th vertex of floor $i$ is:
$$\mathbf{v}_{i,k} = \begin{pmatrix} r(t)\cos(\theta_k + t\Theta) \\ r(t)\sin(\theta_k + t\Theta) \\ t \end{pmatrix}, \quad t = \frac{i}{N_f}, \quad \theta_k = \frac{2\pi k}{n}, \quad k = 0,\ldots,n-1$$
where $r(t) = 1 - \text{taper}\cdot t$ is a linearly tapering radius. Each side face between floors $i$ and $i+1$ is the quadrilateral $\{\mathbf{v}_{i,k},\, \mathbf{v}_{i,k+1},\, \mathbf{v}_{i+1,k+1},\, \mathbf{v}_{i+1,k}\}$.

**Logic in the Code**
1. `floor_verts(i)` computes the $n$ vertex positions at normalised height $t = i/N_f$ using the formula above.
2. For each floor pair $(i,\,i+1)$ and each edge $k \to k{+}1 \pmod{n}$, a quadrilateral face is appended; its centroid height $(i+0.5)/N_f$ sets the face colour via the palette colourmap.
3. All side faces are rendered in one `Poly3DCollection` call. If `show_floors` is enabled, a sparse set of floor-plate polygons is added as a second collection (every $\sim N_f/8$ floors).

**Interesting Properties**
When the total twist equals $2\pi/n$ (one full polygon rotation), the top face is rotationally identical to the bottom, making the twist invisible from directly above — the minimum twist that returns to visual alignment. At $\Theta = \pi$ with an even number of sides, the top and bottom faces are perfectly interleaved, creating a star-profile cross-section at mid-height.

### How This Works — Fractal Mountain

**Core Idea**
The Diamond-Square algorithm generates realistic terrain by hierarchically subdividing a grid and randomly displacing midpoints. The result converges to fractional Brownian motion (fBm) — the same statistical self-similarity found in real mountain profiles and coastlines.

**Mathematics**
Starting from a $(2^n+1)\times(2^n+1)$ grid with random corner heights, the algorithm alternates two steps per level. In the *diamond step*, the centre of each $(s \times s)$ square is set to:
$$h_{\text{mid}} = \tfrac{1}{4}(h_{00} + h_{0s} + h_{s0} + h_{ss}) + \varepsilon\,\sigma_s$$
In the *square step*, each diamond-edge midpoint is set to the average of up to four adjacent values plus $\varepsilon\,\sigma_s$. After each level, the random scale decreases as $\sigma_{s/2} = 2^{-H}\sigma_s$, where $H \in (0,1)$ is the Hurst exponent (`roughness`). The resulting power spectrum is $P(f) \propto f^{-(2H+2)}$, matching natural terrain statistics.

**Logic in the Code**
1. The four corners are randomly initialised; the main loop runs while `step > 1`, halving `step` each iteration and multiplying `scale` by $2^{-H}$.
2. The diamond step fills $(N_f/\text{step})^2$ centre points. The square step iterates over all half-step grid positions and selects unfilled midpoints using the parity check `(y//half + x//half) % 2 == 1`, which correctly identifies diamond-edge midpoints without re-filling already computed values.
3. The normalised height field is passed directly to `ax.plot_surface` as `facecolors`, with an optional translucent sea plane at `sea_level`.

**Interesting Properties**
The roughness parameter $H$ controls the fractal dimension $D = 3 - H$ of the surface: $H=1$ produces smooth rolling hills ($D=2$), while $H \to 0$ produces jagged spiky terrain ($D \to 3$). Real mountain ranges have measured $H \approx 0.5$–0.8. Each distinct `seed` generates an entirely different but statistically equivalent landscape, since the algorithm is a pseudorandom process.

### How This Works — Volumetric Fog Cube

**Core Idea**
Volumetric rendering simulates light scattering through a participating medium (fog, cloud, smoke, nebula). This render approximates a 3D density field using fractional Brownian motion, thresholds it to isolate high-density regions, and plots them as semi-transparent scatter points to simulate a glowing fog cube or nebula.

**Mathematics**
The density field is the sum of $n$ scaled noise octaves:
$$\rho(\mathbf{x}) = \frac{1}{\sum_k 2^{-k}}\sum_{k=0}^{n-1} 2^{-k}\,\eta_k(\mathbf{x})$$
where $\eta_k$ is the $k$-th trilinear-interpolated random noise at frequency $2^k$ (produced by zooming a $(2^{n-k})^3$ random grid up to $N^3$). This 3D fBm has Hurst exponent $H=1$, meaning density varies smoothly at large scales and with increasing fine-grained detail at smaller scales. Voxels with $\rho > \rho_{\min}$ are rendered; their scatter size encodes the excess density $\rho - \rho_{\min}$.

**Logic in the Code**
1. For each octave $k$, a coarse $(2^{n-k})^3$ random array is zoomed to $N^3$ using `scipy.ndimage.zoom` with linear interpolation and accumulated into `field`.
2. A boolean mask `field > density_thresh` selects visible voxels; if the mask is empty (very high threshold), it falls back to the top 50\% of density values.
3. Voxels are sorted back-to-front by $z$-coordinate before calling `ax.scatter`, so near voxels are drawn on top of far voxels, approximating correct alpha compositing without a full ray-marching pipeline. `depthshade=False` preserves colour intensity.

**Interesting Properties**
As `density_thresh` increases, the fog cube transitions from a filled volume to isolated wisps — a 3D percolation threshold. Below the critical density the cloud is one connected region; above it the clusters are isolated blobs. This transition is the visual analogue of bond percolation on a cubic lattice ($p_c \approx 0.249$ for 3D bond percolation), and is responsible for the sudden ‘vanishing’ of large connected cloud structures as `density_thresh` crosses the threshold.

### How This Works — Strange Attractor 3D

**Core Idea**
A strange attractor is the long-time trajectory of a chaotic dynamical system that never repeats yet stays bounded, tracing a fractal structure in 3D phase space. This render integrates four classic 3D systems — Lorenz, Rössler, Thomas’, and Halvorsen — using 4th-order Runge-Kutta and plots the trajectory as a gradient-coloured line.

**Mathematics**
Each attractor is a 3D autonomous ODE $\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x})$. The Halvorsen attractor, for example:
$$\dot{x} = -ax - 4y - 4z - y^2, \quad \dot{y} = -ay - 4z - 4x - z^2, \quad \dot{z} = -az - 4x - 4y - x^2, \quad a = 1.4$$
All four systems have sensitive dependence on initial conditions: two trajectories starting $\varepsilon$ apart diverge exponentially at rate $\lambda_{\max} > 0$ (positive largest Lyapunov exponent). The trajectory never intersects itself, never converges to a fixed point or limit cycle, yet fills a bounded fractal set in $\mathbb{R}^3$.

**Logic in the Code**
1. One of four ODE systems is selected by `attractor` (0 = Lorenz, 1 = Rössler, 2 = Thomas’, 3 = Halvorsen). Each uses a per-attractor fixed `dt` chosen for stability and sufficient orbit coverage.
2. The trajectory is integrated in a Python RK4 loop: $k_1$–$k_4$ are computed per step, with the update $\mathbf{p} \mathrel{+}= \tfrac{dt}{6}(k_1 + 2k_2 + 2k_3 + k_4)$.
3. The first 5\% of steps are discarded as initial transient; per-axis 1–99 percentile clipping removes any outlier excursions that would distort the view.
4. The trajectory is split into $(N-1)$ unit segments and rendered as a single `Line3DCollection`; colour maps position-along-trajectory from palette start to end, making the winding order visible.

**Interesting Properties**
The Lorenz attractor has fractal dimension $D \approx 2.06$ — barely more than a surface. The Rössler attractor is the simplest: one of its Lyapunov exponents is zero, so the attractor collapses to a nearly 2D sheet in one direction, visible as the flat spiral structure. Thomas’ cyclically symmetric attractor ($b = 0.208$) has identical dynamics under $120^\circ$ rotation in $(x, y, z)$ space, giving it a three-fold symmetry that is visible in the plot.

---

## Section 4-E — Scientific & Simulation Patterns (91–100)

Physics simulations, biological models, and computational science visualisations: neural networks, quantum mechanics, general relativity, cellular automata, and collective behaviour.

### How This Works — Neural Network Visualization

**Core Idea**
A feedforward neural network maps an input vector through successive layers of weighted linear combinations and non-linear activations to produce an output. This render draws the network graph: nodes coloured by activation value and edges coloured and weighted by connection strength, revealing the information-flow geometry of the architecture.

**Mathematics**
Each neuron $j$ in layer $l+1$ computes:
$$a_j^{(l+1)} = \sigma\!\left(\sum_i w_{ij}^{(l)}\, a_i^{(l)} + b_j^{(l)}\right)$$
where $w_{ij}^{(l)}$ is the weight from neuron $i$ to $j$, $b_j^{(l)}$ is the bias, and $\sigma$ is a non-linear activation function. The architecture is fully specified by the layer-width sequence $[n_0, n_1, \ldots, n_L]$, set via `layer_sizes`.

**Logic in the Code**
1. `layer_sizes` is parsed into a list of integers; up to 8 layers are supported.
2. Node positions are placed on a uniform $x$-grid with nodes vertically centred and evenly spaced within each column.
3. All $n_i \times n_{i+1}$ edges per adjacent layer pair are batched into a single `LineCollection`; colour maps the randomly sampled weight $w \in [-1, 1]$ (warm = positive, cool = negative) and alpha scales with $|w|$.
4. Each node is a filled `Circle` coloured by activation; a low-alpha glow ring creates the neon halo effect.

**Interesting Properties**
Narrow hidden layers (e.g. `"4,2,2,4"`) form a bottleneck architecture as in autoencoders, forcing information compression. Wide single-hidden-layer networks are universal function approximators: by the Universal Approximation Theorem, sufficient width suffices to approximate any continuous function on a compact domain to arbitrary precision.

### How This Works — Atom Orbital Simulator

**Core Idea**
Electron orbitals are the probability density distributions $|\psi_{nlm}|^2$ of a hydrogen-like atom, derived by solving the Schrödinger equation in a Coulomb potential. Each orbital is characterised by quantum numbers $(n, l, m)$ and has a distinctive lobe or ring structure that determines chemical bonding.

**Mathematics**
The hydrogen wavefunction separates in spherical coordinates:
$$\psi_{nlm}(r, \theta, \phi) = R_{nl}(r)\, Y_l^m(\theta, \phi)$$
The radial part is:
$$R_{nl}(r) \propto e^{-\rho/2}\, \rho^l\, L_{n-l-1}^{2l+1}(\rho),\quad \rho = \frac{2r}{n a_0}$$
where $L_p^q$ is the associated Laguerre polynomial and $Y_l^m$ is the spherical harmonic. Real orbital shapes use $\cos(m\phi)$ and $\sin(m\phi)$ combinations for $m \neq 0$.

**Logic in the Code**
1. A 2D Cartesian grid is built in the chosen viewing plane (xz, xy, or yz) with the third coordinate set to zero.
2. Grid points are converted to spherical coordinates $(r, \theta, \phi)$.
3. `scipy.special.assoc_laguerre` evaluates $L_{n-l-1}^{2l+1}(\rho)$ and `sph_harm` evaluates $Y_l^m$; their product gives $\psi$.
4. $|\psi|^2$ is normalised to $[0,1]$ and rendered as an `imshow` heatmap.

**Interesting Properties**
The total number of nodal surfaces is $n - 1$: $n - l - 1$ radial spherical nodes and $l$ angular nodes. The 3s orbital ($n=3, l=0$) has two concentric spherical zero-probability shells; the 3d orbital ($n=3, l=2$) has no radial nodes and the iconic double-dumbbell or four-lobed cloverleaf shape depending on $m$.

### How This Works — Black Hole Lensing

**Core Idea**
A black hole curves spacetime so severely that light bends around it. A distant observer sees background stars displaced, source rings distorted into Einstein arcs, and a glowing accretion disk whose inner edge sits at the innermost stable circular orbit.

**Mathematics**
In the Schwarzschild metric (non-rotating black hole, $G = c = 1$), the Schwarzschild radius is $r_s = 2M$. The weak-field deflection angle for a photon with impact parameter $b$ is:
$$\hat{\alpha} = \frac{4M}{b}$$
The photon sphere sits at $r_\text{ph} = \tfrac{3}{2}r_s$ and the ISCO at $r_\text{ISCO} = 3r_s$. The apparent position of a source ring at radius $r_\text{src}$ is:
$$r_\text{app} \approx \frac{r_\text{src}}{1 + 4M/r_\text{src}}$$

**Logic in the Code**
1. A random star field is scattered uniformly; stars inside the event horizon are removed.
2. `n_rings` source circles are displaced inward by the weak-field formula and drawn as concentric coloured rings.
3. The accretion disk is a scatter in $[r_\text{ISCO}, 6M]$; brightness follows $I \propto (r_\text{ISCO}/r)^{2.5}$ with multiplicative noise for texture.
4. The event horizon is a solid black circle; the photon sphere is an optional dashed orange ring at $1.5\,r_s$.

**Interesting Properties**
The Event Horizon Telescope image of M87* (2019) shows a bright ring at $\sim 3r_s$ and a dark shadow inside, matching Schwarzschild predictions. For a rotating Kerr black hole the shadow is asymmetric: the approaching disk side is Doppler-boosted and brighter, while the receding side is dimmer, creating the crescent seen in real black hole images.

### How This Works — Conway's Game of Life

**Core Idea**
Conway's Game of Life (1970) is a zero-player cellular automaton on an infinite 2D grid. Four deterministic rules applied simultaneously each generation produce rich emergent behaviour — still lifes, oscillators, gliders, and universal computation — from a dead-simple local neighbourhood rule.

**Mathematics**
Let $g(i,j,t) \in \{0,1\}$ be cell state and $N(i,j,t)$ its 8-cell Moore neighbourhood sum. The update rule is:
$$g(i,j,t+1) = \begin{cases} 1 & g=1,\ N \in \{2,3\} \ (\text{survival}) \\ 1 & g=0,\ N = 3 \ (\text{birth}) \\ 0 & \text{otherwise}\end{cases}$$

**Logic in the Code**
1. The grid is initialised from one of four seeded patterns: uniform random at density $p$, two gliders, the R-pentomino (a 5-cell methuselah with 1 105-generation lifespan), or the Gosper glider gun.
2. Each generation, `scipy.ndimage.convolve` with a $3\times 3$ all-ones kernel (centre=0) computes the Moore neighbourhood sum in one vectorised operation.
3. Birth and survival masks are combined into the next grid state.
4. A per-cell age counter increments for live cells and resets on death; log-normalised age is mapped through the colormap so newly born cells glow brightest.

**Interesting Properties**
Conway's Game of Life is Turing-complete. The R-pentomino stabilises after 1 105 generations, producing six gliders, eight blocks, four blinkers, and several still lifes. The density phase transition near $p \approx 0.37$ separates sparse dying populations from dense oscillating ones.